In [1]:
import os
import sys
import re
import numpy as np
import pandas as pd
import plotly.express as px
import scipy.stats as stats
from scipy.stats import pearsonr
from dash import html, dcc, Input, Output, Dash
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.append(os.path.join(os.getcwd(), '..', '..', '..'))
from baseVR.base_functionality import init_import_paths
init_import_paths() 

from CustomLogger import CustomLogger as Logger
from analytics_processing import analytics

from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args, fullfnames2snames
from dashsrc.plot_components.plots import plot_TrackFiringRate
from dashsrc.plot_components.plots import plot_unit_fr_stability


In [2]:
Logger().init_logger(None, None, logging_level="DEBUG")
animal_ids = [6]
paradigm = [1100]
session_range = [1,33]
session_ids = None
normalize = True
smooth = False
excl_session_names = ['2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min', '2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min', '2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min']

session_dirs = sessionlist_fullfnames_from_args(paradigm, animal_ids, session_ids, excl_session_names=excl_session_names)[0]
session_names= fullfnames2snames(session_dirs)

2026-03-06 12:41:09,830|DEBUG|2193354|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Searching NAS for applicable sessions...
2026-03-06 12:41:09,831|DEBUG|2193354|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min.hdf5 excluded
2026-03-06 12:41:09,831|DEBUG|2193354|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2024-12-11_17-42_rYL006_P1100_LinearTrackStop_30min.hdf5 excluded
2026-03-06 12:41:09,832|DEBUG|2193354|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min.hdf5 excluded
2026-03-06 12:41:09,832|DEBUG|2193354|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min.hdf5 excluded
2026-03-06 12:41:09,833|DEBUG|2193354|sessions_from_nas_parsing|get_sessionlist_fullfnames
	For paradigms [1100], animals [6], found 30 sessions.
2026-03-06 12:41:09,833|DEBUG|2193354|sessions_from_nas_pa

In [3]:
# firing rates and behavior data
fr = analytics.get_analytics('FiringRate40msHz', session_names=session_names)
# fr_track = analytics.get_analytics('FiringRateTrackwiseHz', session_names=session_names)
t0_events = analytics.get_analytics('TrialWiseT0Events40ms', session_names=session_names)
fr_z_scored  = analytics.get_analytics('FiringRate40msZ', session_names=session_names)
fr_z_all_sess = fr.apply(lambda unit_fr: ((unit_fr - unit_fr.mean()) / unit_fr.std()))
# behav = analytics.get_analytics('BehaviorTrackwise', session_names=session_names)
# behav.index = behav.index.droplevel(('animal_id', 'paradigm_id', 'entry_id'))

2026-03-06 12:41:09,838|DEBUG|2193354|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-03-06 12:41:09,839|DEBUG|2193354|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-03-06 12:41:09,839|DEBUG|2193354|analytics|get_analytics
	Processing FiringRate40msHz, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/mnt/slow/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-03-06 12:41:09,840|INFO|2193354|analytics|get_analytics
	Analytic `FiringRate40msHz` does not exist for (1100, 6, '2024-11-14_15-01'), compute first, or check for typo
2026-03-06 12:41:09,840|DEBUG|2193354|analytics|get_analytics
	Processing FiringRate40msHz, (1100, 6, '2024-11-14_16-40') 2024-11-14_16-40_rYL006_P1100_LinearTrackStop_21min

In [4]:
t0_events

trial_id  cue  trial_outcome  \
paradigm_id animal_id session_id       entry_id                                 
1100        6         2024-11-14_16-40 0              1.0  1.0            0.0   
                                       1              1.0  1.0            0.0   
                                       2              1.0  1.0            0.0   
                                       3              1.0  1.0            0.0   
                                       4              1.0  1.0            0.0   
...                                                   ...  ...            ...   
                      2025-01-27_13-39 1104         152.0  2.0            1.0   
                                       1105         152.0  2.0            1.0   
                                       1106         152.0  2.0            1.0   
                                       1107         152.0  2.0            1.0   
                                       1108         152.0  2.0            1.0   

                                                 choice_R1  choice_R2  \
paradigm_id animal_id session_id       entry_id                         
1100        6         2024-11-14_16-40 0               0.0        0.0   
                                       1               0.0        0.0   
                                       2               0.0        0.0   
                                       3               0.0        0.0   
                                       4               0.0        0.0   
...                                                    ...        ...   
                      2025-01-27_13-39 1104            0.0        1.0   
                                       1105            0.0        1.0   
                                       1106            0.0        1.0   
                                       1107            0.0        1.0   
                                       1108            0.0        1.0   

                                                     t0_event_name  \
paradigm_id animal_id session_id       entry_id                      
1100        6         2024-11-14_16-40 0           cueZone_visible   
                                       1             cueZone_entry   
                                       2              cueZone_exit   
                                       3         enter_reward1Zone   
                                       4         enter_reward2Zone   
...                                                            ...   
                      2025-01-27_13-39 1104           cueZone_exit   
                                       1105      enter_reward1Zone   
                                       1106      enter_reward2Zone   
                                       1107       exit_reward1Zone   
                                       1108       exit_reward2Zone   

                                                         t0  x_position  \
paradigm_id animal_id session_id       entry_id                           
1100        6         2024-11-14_16-40 0            4600000 -119.732067   
                                       1            5400000  -79.536247   
                                       2            6760000   24.898020   
                                       3            7080000   51.438920   
                                       4            9480000  170.376450   
...                                                     ...         ...   
                      2025-01-27_13-39 1104      4396000000   25.843260   
                                       1105      4396480000   49.770475   
                                       1106      4398880000  170.561500   
                                       1107      4397720000  109.711900   
                                       1108      4401200000  229.449700   

                                                 x_alignment  \
paradigm_id animal_id session_id       entry_id                
1100        6         2024-11-

In [5]:
# check whether sessions in both dfs match
session_ids_fr = set(t0_events.index.get_level_values("session_id").dropna().unique())
session_ids_t0 = set(fr.index.get_level_values("session_id").dropna().unique())
sessions_queal = session_ids_fr == session_ids_t0
print(f"Session IDs in fr: {len(session_ids_fr)}")
print(f"Session IDs in t0: {len(session_ids_t0)}")
print(f"Session IDs in both: {len(session_ids_fr & session_ids_t0)}")
print(f"Session IDs are equal: {sessions_queal}")

Session IDs in fr: 25
Session IDs in t0: 25
Session IDs in both: 25
Session IDs are equal: True


In [6]:
# Event-based neuron classification: cue / choice / reward_sound (+ detailed event regions)
# Tests whether units fire more during event intervals than during the rest of session time.

N_SHUFFLES = 1000
THRESHOLD_PERCENTILE = 99
ALPHA = 0.01
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

def _parse_interval(value):
    if isinstance(value, pd.Interval):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    if isinstance(value, dict):
        left = value.get("left", np.nan)
        right = value.get("right", np.nan)
        if pd.notna(left) and pd.notna(right):
            return pd.Interval(float(left), float(right), closed="right")
        return np.nan
    if isinstance(value, (tuple, list)) and len(value) >= 2:
        left, right = value[0], value[1]
        if pd.notna(left) and pd.notna(right):
            return pd.Interval(float(left), float(right), closed="right")
        return np.nan

    txt = str(value).strip()
    if txt.lower() in {"", "nan", "none", "nat"}:
        return np.nan

    m = re.match(r"^[\(\[]\s*([-+0-9eE\.]+)\s*,\s*([-+0-9eE\.]+)\s*[\)\]]$", txt)
    if m:
        return pd.Interval(float(m.group(1)), float(m.group(2)), closed="right")

    return np.nan


def _safe_divide(num, den):
    return np.divide(
        num,
        den,
        out=np.full_like(num, np.nan, dtype=float),
        where=np.asarray(den) > 0,
    )


def _compute_diff_from_idx(x_data, idx, total_sum, total_cnt, valid_mask):
    n_rows = x_data.shape[0]
    if idx.size == 0 or idx.size >= n_rows:
        return np.full(x_data.shape[1], np.nan, dtype=float)

    in_sum = x_data[idx, :].sum(axis=0)
    if valid_mask is None:
        in_cnt = np.full(x_data.shape[1], idx.size, dtype=float)
    else:
        in_cnt = valid_mask[idx, :].sum(axis=0)

    out_sum = total_sum - in_sum
    out_cnt = total_cnt - in_cnt

    in_mean = _safe_divide(in_sum, in_cnt)
    out_mean = _safe_divide(out_sum, out_cnt)
    return in_mean - out_mean


fr_flat = fr.reset_index().copy()
t0_flat = t0_events.reset_index().copy()

if {"from_ephys_timestamp", "to_ephys_timestamp"}.issubset(fr_flat.columns):
    fr_flat["__time_mid"] = (
        pd.to_numeric(fr_flat["from_ephys_timestamp"], errors="coerce")
        + pd.to_numeric(fr_flat["to_ephys_timestamp"], errors="coerce")
    ) / 2.0
elif "from_ephys_timestamp" in fr_flat.columns:
    fr_flat["__time_mid"] = pd.to_numeric(fr_flat["from_ephys_timestamp"], errors="coerce")

fr_flat = fr_flat.dropna(subset=["session_id", "__time_mid"]).copy()
unit_cols = [c for c in fr_flat.columns if str(c).startswith("Unit")]

trial_col = next((c for c in ["trial_id", "triald_id"] if c in t0_flat.columns), None)
if trial_col is None:
    trial_col = "trial_id"
    t0_flat[trial_col] = np.nan

raw_interval_candidates = {
    "cue_entry": ["cue_entry_interval"],
    "reward1_sound": ["reward1_sound_interval", "reward1_valve_open_interval"],
    "reward2_sound": ["reward2_sound_interval", "reward2_valve_open_interval"],
    "R1_entry": ["R1_entry_interval"],
    "R2_entry": ["R2_entry_interval"],
}

resolved_interval_cols = {}
for key, candidates in raw_interval_candidates.items():
    col = next((c for c in candidates if c in t0_flat.columns), None)
    if col is not None:
        resolved_interval_cols[key] = col

parsed_col_map = {}
for key, col in resolved_interval_cols.items():
    parsed_col = f"__parsed_{key}"
    t0_flat[parsed_col] = t0_flat[col].map(_parse_interval)
    parsed_col_map[key] = parsed_col

session_ids = pd.Index(fr_flat["session_id"].dropna().unique()).intersection(
    pd.Index(t0_flat["session_id"].dropna().unique())
)

records = []
timing_records = []
timing_mismatch_examples = []

for session_id in session_ids:
    fr_s = (
        fr_flat[fr_flat["session_id"] == session_id]
        .sort_values("__time_mid")
        .reset_index(drop=True)
    )
    t0_s = t0_flat[t0_flat["session_id"] == session_id].copy()

    if fr_s.empty or t0_s.empty:
        continue

    time_mid = fr_s["__time_mid"].to_numpy(dtype=float)
    n_rows = len(fr_s)
    n_units = len(unit_cols)

    raw_masks = {}
    for key, parsed_col in parsed_col_map.items():
        mask = np.zeros(n_rows, dtype=bool)
        valid_rows = t0_s.loc[t0_s[parsed_col].notna(), [trial_col, parsed_col]].copy()

        n_intervals = len(valid_rows)
        n_matched = 0

        for i, intvl in enumerate(valid_rows[parsed_col].to_numpy()):
            left = float(intvl.left)
            right = float(intvl.right)
            lo = int(np.searchsorted(time_mid, left, side="left"))
            hi = int(np.searchsorted(time_mid, right, side="right"))

            if hi > lo:
                mask[lo:hi] = True
                n_matched += 1
            else:
                if len(timing_mismatch_examples) < 30:
                    timing_mismatch_examples.append(
                        {
                            "session_id": session_id,
                            "interval_key": key,
                            "trial_id": valid_rows.iloc[i][trial_col],
                            "interval_left": left,
                            "interval_right": right,
                            "fr_min_time": float(time_mid.min()),
                            "fr_max_time": float(time_mid.max()),
                        }
                    )

        raw_masks[key] = mask
        timing_records.append(
            {
                "session_id": session_id,
                "interval_key": key,
                "interval_column": resolved_interval_cols[key],
                "n_intervals_total": int(n_intervals),
                "n_intervals_matched": int(n_matched),
                "n_intervals_unmatched": int(n_intervals - n_matched),
                "pct_intervals_matched": float(100 * n_matched / max(n_intervals, 1)),
                "n_bins_covered": int(mask.sum()),
                "n_bins_session": int(n_rows),
                "fr_time_min": float(time_mid.min()),
                "fr_time_max": float(time_mid.max()),
            }
        )

    feature_masks = {
        "cue": raw_masks.get("cue_entry", np.zeros(n_rows, dtype=bool)),
        "choice": (
            raw_masks.get("R1_entry", np.zeros(n_rows, dtype=bool))
            | raw_masks.get("R2_entry", np.zeros(n_rows, dtype=bool))
        ),
        "reward_sound": (
            raw_masks.get("reward1_sound", np.zeros(n_rows, dtype=bool))
            | raw_masks.get("reward2_sound", np.zeros(n_rows, dtype=bool))
        ),
        "choice_r1": raw_masks.get("R1_entry", np.zeros(n_rows, dtype=bool)),
        "choice_r2": raw_masks.get("R2_entry", np.zeros(n_rows, dtype=bool)),
        "reward_sound_r1": raw_masks.get("reward1_sound", np.zeros(n_rows, dtype=bool)),
        "reward_sound_r2": raw_masks.get("reward2_sound", np.zeros(n_rows, dtype=bool)),
    }
    feature_order = list(feature_masks.keys())
    feature_idx = {k: np.flatnonzero(v) for k, v in feature_masks.items()}

    x_raw = fr_s[unit_cols].to_numpy(dtype=float, copy=True)
    has_nan = np.isnan(x_raw).any()
    if has_nan:
        valid_mask = np.isfinite(x_raw)
        x_data = np.where(valid_mask, x_raw, 0.0)
        total_sum = x_data.sum(axis=0)
        total_cnt = valid_mask.sum(axis=0).astype(float)
    else:
        valid_mask = None
        x_data = x_raw
        total_sum = x_data.sum(axis=0)
        total_cnt = np.full(n_units, n_rows, dtype=float)

    observed = {}
    shuffled = {k: np.full((N_SHUFFLES, n_units), np.nan, dtype=float) for k in feature_order}

    for key in feature_order:
        observed[key] = _compute_diff_from_idx(
            x_data=x_data,
            idx=feature_idx[key],
            total_sum=total_sum,
            total_cnt=total_cnt,
            valid_mask=valid_mask,
        )

    if n_rows > 1:
        for s in range(N_SHUFFLES):
            shift = int(rng.integers(0, n_rows))
            for key in feature_order:
                base_idx = feature_idx[key]
                if base_idx.size == 0 or base_idx.size >= n_rows:
                    continue
                shifted_idx = (base_idx + shift) % n_rows
                shuffled[key][s, :] = _compute_diff_from_idx(
                    x_data=x_data,
                    idx=shifted_idx,
                    total_sum=total_sum,
                    total_cnt=total_cnt,
                    valid_mask=valid_mask,
                )

    for u_i, unit in enumerate(unit_cols):
        rec = {
            "session_id": session_id,
            "unit": unit,
            "n_rows_session": int(n_rows),
        }

        for key in feature_order:
            rec[f"n_rows_{key}"] = int(feature_idx[key].size)

            obs = observed[key][u_i]
            sh = shuffled[key][:, u_i]
            valid = np.isfinite(sh)

            if valid.any() and np.isfinite(obs):
                thr = float(np.percentile(sh[valid], THRESHOLD_PERCENTILE))
                pval = float((np.sum(sh[valid] >= obs) + 1) / (valid.sum() + 1))
            else:
                thr = np.nan
                pval = np.nan

            rec[f"obs_{key}"] = float(obs) if np.isfinite(obs) else np.nan
            rec[f"thr99_{key}"] = thr
            rec[f"p_{key}"] = pval
            rec[f"is_{key}_cell"] = bool(np.isfinite(obs) and np.isfinite(thr) and (obs > thr))

        labels = []
        if rec["is_cue_cell"]:
            labels.append("cue")
        if rec["is_choice_cell"]:
            labels.append("choice")
        if rec["is_reward_sound_cell"]:
            labels.append("reward_sound")
        rec["assigned_label"] = "+".join(labels) if labels else "none"

        records.append(rec)

classification_eventwise = (
    pd.DataFrame(records)
    .sort_values(["session_id", "unit"])
    .reset_index(drop=True)
)

timing_sanity = pd.DataFrame(timing_records)
timing_summary = (
    timing_sanity.groupby("interval_key", as_index=False)
    .agg(
        n_sessions=("session_id", "nunique"),
        n_intervals_total=("n_intervals_total", "sum"),
        n_intervals_matched=("n_intervals_matched", "sum"),
        n_intervals_unmatched=("n_intervals_unmatched", "sum"),
        n_bins_covered=("n_bins_covered", "sum"),
    )
)
timing_summary["pct_intervals_matched"] = (
    100 * timing_summary["n_intervals_matched"] / timing_summary["n_intervals_total"].clip(lower=1)
)

timing_mismatches = pd.DataFrame(timing_mismatch_examples)

print("Resolved interval columns:", resolved_interval_cols)
print(
    f"Timing sanity check across {timing_sanity['session_id'].nunique()} sessions "
    f"(FR rows: {len(fr_flat):,}, t0 rows: {len(t0_flat):,})"
)
display(timing_summary.sort_values("interval_key"))
if not timing_mismatches.empty:
    print("Examples of unmatched event intervals (first 30):")
    display(timing_mismatches)

broad_flags = ["is_cue_cell", "is_choice_cell", "is_reward_sound_cell"]
detail_flags = [
    "is_choice_r1_cell",
    "is_choice_r2_cell",
    "is_reward_sound_r1_cell",
    "is_reward_sound_r2_cell",
]

print(
    f"Computed classification for {len(classification_eventwise)} unit-session entries "
    f"({classification_eventwise['unit'].nunique()} units, "
    f"{classification_eventwise['session_id'].nunique()} sessions)."
)
display(classification_eventwise[broad_flags + detail_flags].sum().rename("n_unit_sessions").to_frame())
display(classification_eventwise.head())

# Trackwise-like plots (renamed reward_approach -> choice)
dfc = classification_eventwise.copy()

features = ["cue", "choice", "reward_sound"]
obs_cols = [f"obs_{f}" for f in features]
p_cols = [f"p_{f}" for f in features]

obs = dfc[obs_cols].to_numpy(dtype=float)
pvals = dfc[p_cols].to_numpy(dtype=float)

obs_filled = np.where(np.isfinite(obs), obs, -np.inf)
has_obs = np.isfinite(obs).any(axis=1)

pref_idx = np.argmax(obs_filled, axis=1)
pref_obs = np.where(has_obs, obs[np.arange(len(dfc)), pref_idx], np.nan)

runner = obs_filled.copy()
runner[np.arange(len(dfc)), pref_idx] = -np.inf
runner_obs = np.max(runner, axis=1)
runner_obs = np.where(has_obs & np.isfinite(runner_obs), runner_obs, np.nan)

selectivity_index = (pref_obs - runner_obs) / (
    np.abs(pref_obs) + np.abs(runner_obs) + 1e-12
)

pref_p = np.where(has_obs, pvals[np.arange(len(dfc)), pref_idx], np.nan)
shuffle_confidence = 1 - pref_p
is_significant_pref = pref_p < ALPHA
preferred_feature = np.where(
    has_obs,
    np.array(features, dtype=object)[pref_idx],
    "none",
)

certainty_df = dfc[["session_id", "unit"]].copy()
certainty_df["preferred_feature"] = preferred_feature
certainty_df["pref_obs"] = pref_obs
certainty_df["runner_obs"] = runner_obs
certainty_df["selectivity_index"] = selectivity_index
certainty_df["pref_p"] = pref_p
certainty_df["shuffle_confidence"] = shuffle_confidence
certainty_df["certainty_score"] = certainty_df["selectivity_index"] * certainty_df["shuffle_confidence"]
certainty_df["is_significant_pref"] = is_significant_pref

print(
    f"unit-session rows: {len(certainty_df)} | unique units: {certainty_df['unit'].nunique()} | "
    f"unique sessions: {certainty_df['session_id'].nunique()}"
)

fig_certainty = px.scatter(
    certainty_df,
    x="selectivity_index",
    y="shuffle_confidence",
    color="preferred_feature",
    symbol="is_significant_pref",
    opacity=0.7,
    hover_data={
        "session_id": True,
        "unit": True,
        "pref_obs": ":.4f",
        "runner_obs": ":.4f",
        "pref_p": ":.4g",
        "certainty_score": ":.4f",
    },
    title="Event-encoding certainty per unit-session (selectivity vs shuffle confidence)",
    labels={
        "selectivity_index": "Selectivity vs next-best event",
        "shuffle_confidence": "Shuffle confidence (1 - p)",
        "preferred_feature": "Preferred encoding",
        "is_significant_pref": "p < 0.01",
    },
)
fig_certainty.update_layout(template="plotly_white")
fig_certainty.show()

flag_map = {
    "is_cue_cell": "cue",
    "is_choice_cell": "choice",
    "is_reward_sound_cell": "reward_sound",
}
session_frac = (
    dfc.groupby("session_id", as_index=False)[list(flag_map.keys())]
    .mean()
    .rename(columns=flag_map)
)

session_long = session_frac.melt(
    id_vars="session_id",
    var_name="cell_type",
    value_name="fraction_units",
)
session_long["percent_units"] = 100.0 * session_long["fraction_units"]

session_long["session_dt"] = pd.to_datetime(session_long["session_id"], errors="coerce")
if session_long["session_dt"].notna().any():
    session_long = session_long.sort_values(["session_dt", "cell_type"])
else:
    session_long = session_long.sort_values(["session_id", "cell_type"])

fig_dynamics = px.line(
    session_long,
    x="session_id",
    y="percent_units",
    color="cell_type",
    markers=True,
    title="Event-encoding prevalence across sessions (% classified units)",
    labels={
        "session_id": "Session",
        "percent_units": "% of units",
        "cell_type": "Encoding type",
    },
)
fig_dynamics.update_layout(template="plotly_white")
fig_dynamics.update_yaxes(range=[0, 100])
fig_dynamics.show()

detail_flag_map = {
    "is_choice_r1_cell": "choice_r1",
    "is_choice_r2_cell": "choice_r2",
    "is_reward_sound_r1_cell": "reward_sound_r1",
    "is_reward_sound_r2_cell": "reward_sound_r2",
}
detail_frac = (
    dfc.groupby("session_id", as_index=False)[list(detail_flag_map.keys())]
    .mean()
    .rename(columns=detail_flag_map)
)

detail_long = detail_frac.melt(
    id_vars="session_id",
    var_name="event_type",
    value_name="fraction_units",
)
detail_long["percent_units"] = 100.0 * detail_long["fraction_units"]

detail_long["session_dt"] = pd.to_datetime(detail_long["session_id"], errors="coerce")
if detail_long["session_dt"].notna().any():
    detail_long = detail_long.sort_values(["session_dt", "event_type"])
else:
    detail_long = detail_long.sort_values(["session_id", "event_type"])

fig_detail = px.line(
    detail_long,
    x="session_id",
    y="percent_units",
    color="event_type",
    markers=True,
    title="Detailed event-region prevalence across sessions (% classified units)",
    labels={
        "session_id": "Session",
        "percent_units": "% of units",
        "event_type": "Detailed event type",
    },
)
fig_detail.update_layout(template="plotly_white")
fig_detail.update_yaxes(range=[0, 100])
fig_detail.show()




Resolved interval columns: {'cue_entry': 'cue_entry_interval', 'reward1_sound': 'reward1_sound_interval', 'reward2_sound': 'reward2_sound_interval', 'R1_entry': 'R1_entry_interval', 'R2_entry': 'R2_entry_interval'}
Timing sanity check across 25 sessions (FR rows: 1,272,389, t0 rows: 19,722)


,interval_key,n_sessions,n_intervals_total,n_intervals_matched,n_intervals_unmatched,n_bins_covered,pct_intervals_matched
0,R1_entry,25,2716,2716,0,108640,100.0
1,R2_entry,25,2716,2716,0,108640,100.0
2,cue_entry,25,2716,2716,0,108640,100.0
3,reward1_sound,25,378,378,0,15120,100.0
4,reward2_sound,25,332,332,0,13280,100.0


Computed classification for 1925 unit-session entries (77 units, 25 sessions).


,n_unit_sessions
is_cue_cell,278
is_choice_cell,356
is_reward_sound_cell,249
is_choice_r1_cell,317
is_choice_r2_cell,268
is_reward_sound_r1_cell,156
is_reward_sound_r2_cell,152


,session_id,unit,n_rows_session,n_rows_cue,obs_cue,thr99_cue,p_cue,is_cue_cell,n_rows_choice,obs_choice,...,obs_reward_sound_r1,thr99_reward_sound_r1,p_reward_sound_r1,is_reward_sound_r1_cell,n_rows_reward_sound_r2,obs_reward_sound_r2,thr99_reward_sound_r2,p_reward_sound_r2,is_reward_sound_r2_cell,assigned_label
0,2024-11-14_16-40,Unit0001,32200,4080,-1.012572,0.341612,1.000000,False,8154,0.056505,...,0.893758,1.105244,0.036963,False,120,-0.161835,2.347517,0.576424,False,none
1,2024-11-14_16-40,Unit0002,32200,4080,-0.209179,0.141646,1.000000,False,8154,0.293518,...,0.144231,0.408588,0.259740,False,120,-0.275613,0.979063,0.886114,False,choice
2,2024-11-14_16-40,Unit0003,32200,4080,-0.006119,0.099199,0.579421,False,8154,0.091598,...,0.087878,0.299364,0.280719,False,120,0.191448,0.611765,0.315684,False,choice
3,2024-11-14_16-40,Unit0004,32200,4080,-0.009082,0.033017,0.849151,False,8154,0.014025,...,-0.020492,0.138123,1.000000,False,120,0.188851,0.188851,0.094905,False,none
4,2024-11-14_16-40,Unit0005,32200,4080,-0.375239,0.179135,1.000000,False,8154,0.029515,...,0.220944,0.643916,0.151848,False,120,-0.408874,1.264027,0.916084,False,none


unit-session rows: 1925 | unique units: 77 | unique sessions: 25


/tmp/ipykernel_2193354/3394360912.py:414: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



/tmp/ipykernel_2193354/3394360912.py:456: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



In [7]:
# Region-level and longitudinal plots (trackwise-style) + evidence heatmap
MIN_SIG_SESSIONS_STABLE = 3
features = ["cue", "choice", "reward_sound"]

dfc = classification_eventwise.copy()


def _normalize_unit_from_series(series: pd.Series) -> pd.Series:
    txt = series.astype(str).str.strip()
    from_label = txt.str.extract(r"(?i)unit\s*0*(\d+)", expand=False)
    from_label_num = pd.to_numeric(from_label, errors="coerce")
    from_num = pd.to_numeric(series, errors="coerce")
    unit_num = from_label_num.combine_first(from_num)
    return unit_num.map(lambda v: f"Unit{int(v):04d}" if pd.notna(v) else np.nan)


def _build_meta_map_from_spike(spike_df: pd.DataFrame) -> pd.DataFrame:
    if spike_df is None or len(spike_df) == 0:
        return pd.DataFrame(columns=["session_id", "unit", "brain_region"])

    mdf = spike_df.reset_index().copy()
    if "session_id" not in mdf.columns:
        return pd.DataFrame(columns=["session_id", "unit", "brain_region"])

    area_candidates = ["fine_brain_area", "brain_area", "region", "area"]
    area_col = next((c for c in area_candidates if c in mdf.columns), None)
    if area_col is None:
        mdf["brain_region"] = "Unknown"
    else:
        mdf["brain_region"] = (
            mdf[area_col]
            .astype(str)
            .str.strip()
            .replace({"": np.nan, "nan": np.nan, "None": np.nan})
            .fillna("Unknown")
        )

    unit_col = None
    for candidate in ["unit", "unit_id", "unit_name", "cluster_id", "entry_id"]:
        if candidate in mdf.columns:
            u = _normalize_unit_from_series(mdf[candidate])
            if u.notna().any():
                mdf["unit"] = u
                unit_col = candidate
                break

    if "unit" not in mdf.columns:
        return pd.DataFrame(columns=["session_id", "unit", "brain_region"])

    if unit_col == "entry_id":
        shifted = _normalize_unit_from_series(pd.to_numeric(mdf["entry_id"], errors="coerce") + 1)
        n_match_default = mdf["unit"].isin(dfc["unit"].unique()).sum()
        n_match_shifted = shifted.isin(dfc["unit"].unique()).sum()
        if n_match_shifted > n_match_default:
            mdf["unit"] = shifted

    meta_map = mdf[["session_id", "unit", "brain_region"]].dropna(subset=["session_id", "unit"]).copy()
    meta_map = (
        meta_map.groupby(["session_id", "unit"], as_index=False)["brain_region"]
        .agg(lambda x: x.mode().iat[0] if not x.mode().empty else x.iloc[0])
    )
    return meta_map



meta_spike = analytics.get_analytics("SpikeClusterMetadata", session_names=session_names)

meta_map = _build_meta_map_from_spike(meta_spike)

cert_small = certainty_df[
    [
        "session_id",
        "unit",
        "preferred_feature",
        "is_significant_pref",
        "pref_p",
        "certainty_score",
    ]
].copy()

row_df = (
    dfc.merge(cert_small, on=["session_id", "unit"], how="left")
    .merge(meta_map, on=["session_id", "unit"], how="left")
)
row_df["brain_region"] = row_df["brain_region"].fillna("Unknown")

row_df["encoding_state"] = np.where(
    row_df["is_significant_pref"].fillna(False),
    row_df["preferred_feature"],
    "none",
)

session_order = pd.DataFrame({"session_id": row_df["session_id"].drop_duplicates()})
session_order["session_dt"] = pd.to_datetime(
    session_order["session_id"], format="%Y-%m-%d_%H-%M", errors="coerce"
)
if session_order["session_dt"].notna().any():
    session_order = session_order.sort_values(["session_dt", "session_id"])
else:
    session_order = session_order.sort_values("session_id")
session_order["session_order"] = np.arange(len(session_order))
row_df = row_df.merge(session_order[["session_id", "session_order"]], on="session_id", how="left")


def _dominant_encoding(g: pd.DataFrame) -> str:
    sig = g[g["encoding_state"] != "none"]
    if sig.empty:
        return "none"

    counts = sig["encoding_state"].value_counts()
    top = counts[counts == counts.max()].index.tolist()
    if len(top) == 1:
        return top[0]

    tie = (
        sig[sig["encoding_state"].isin(top)]
        .groupby("encoding_state", as_index=False)["certainty_score"]
        .mean()
        .sort_values("certainty_score", ascending=False)
    )
    return tie.iloc[0]["encoding_state"]


def _unit_summary(g: pd.DataFrame) -> pd.Series:
    sig = g[g["encoding_state"] != "none"]
    n_sig = int(len(sig))
    n_unique = int(sig["encoding_state"].nunique()) if n_sig > 0 else 0

    dom = _dominant_encoding(g)
    if n_sig == 0:
        stability = "unclassified"
    elif n_unique == 1 and n_sig >= MIN_SIG_SESSIONS_STABLE:
        stability = "stable"
    elif n_unique == 1 and n_sig < MIN_SIG_SESSIONS_STABLE:
        stability = "low_support"
    else:
        stability = "changing"

    region_mode = g["brain_region"].mode()
    region = region_mode.iat[0] if not region_mode.empty else g["brain_region"].iloc[0]

    return pd.Series(
        {
            "brain_region": region,
            "dominant_encoding": dom,
            "stability_class": stability,
            "n_sessions": int(g["session_id"].nunique()),
            "n_sig_sessions": n_sig,
            "n_unique_sig_encodings": n_unique,
        }
    )


unit_profile = row_df.groupby("unit", as_index=False).apply(_unit_summary).reset_index(drop=True)

print(
    f"Units total: {len(unit_profile)} | "
    f"stable: {(unit_profile['stability_class'] == 'stable').sum()} | "
    f"changing: {(unit_profile['stability_class'] == 'changing').sum()} | "
    f"unclassified: {(unit_profile['stability_class'] == 'unclassified').sum()}"
)

# Region x dominant encoding
enc_order = ["cue", "choice", "reward_sound", "none"]
unit_profile_no_ca1 = unit_profile[unit_profile["brain_region"] != "CA1"].copy()
region_order = (
    unit_profile_no_ca1["brain_region"].value_counts().sort_values(ascending=False).index.tolist()
)

region_encoding_counts = (
    unit_profile_no_ca1.groupby(["brain_region", "dominant_encoding"], as_index=False)
    .size()
    .rename(columns={"size": "n_units"})
)

fig_region_encoding = px.bar(
    region_encoding_counts,
    x="brain_region",
    y="n_units",
    color="dominant_encoding",
    barmode="stack",
    category_orders={"dominant_encoding": enc_order, "brain_region": region_order},
    title="Neuron counts by brain region and dominant event encoding",
    labels={
        "brain_region": "Brain region",
        "n_units": "Number of neurons",
        "dominant_encoding": "Dominant encoding",
    },
)
fig_region_encoding.update_layout(template="plotly_white", xaxis_tickangle=-35)
fig_region_encoding.show()

# Region x stability
stab_order = ["stable", "changing", "low_support", "unclassified"]
region_stability_counts = (
    unit_profile_no_ca1.groupby(["brain_region", "stability_class"], as_index=False)
    .size()
    .rename(columns={"size": "n_units"})
)

fig_region_stability = px.bar(
    region_stability_counts,
    x="brain_region",
    y="n_units",
    color="stability_class",
    barmode="stack",
    category_orders={"stability_class": stab_order, "brain_region": region_order},
    title="Stable vs changing neurons by brain region (event-based)",
    labels={
        "brain_region": "Brain region",
        "n_units": "Number of neurons",
        "stability_class": "Encoding stability",
    },
)
fig_region_stability.update_layout(template="plotly_white", xaxis_tickangle=-35)
fig_region_stability.show()

# Per-neuron categorical encoding timeline
timeline = row_df[["unit", "session_id", "session_order", "encoding_state", "brain_region"]].drop_duplicates(
    subset=["unit", "session_id"]
)
ordered_sessions = session_order.sort_values("session_order")["session_id"].tolist()

stab_rank = {"changing": 0, "stable": 1, "low_support": 2, "unclassified": 3}
unit_profile_plot = unit_profile.copy()
unit_profile_plot["_stab_rank"] = unit_profile_plot["stability_class"].map(stab_rank).fillna(99)
unit_profile_plot = unit_profile_plot.sort_values(["_stab_rank", "brain_region", "unit"])
unit_order = unit_profile_plot["unit"].tolist()

state_matrix = timeline.pivot(index="unit", columns="session_id", values="encoding_state")
state_matrix = state_matrix.reindex(index=unit_order, columns=ordered_sessions)

state_to_int = {"none": 0, "cue": 1, "choice": 2, "reward_sound": 3}
z_state = state_matrix.replace(state_to_int).fillna(0).to_numpy()

label_map = unit_profile_plot.set_index("unit").apply(
    lambda r: f"{r.name} | {r['brain_region']} | {r['stability_class']}", axis=1
)
y_labels = [label_map.get(u, u) for u in state_matrix.index]

state_colorscale = [
    [0.00, "#d9d9d9"], [0.24, "#d9d9d9"],
    [0.25, "#1f77b4"], [0.49, "#1f77b4"],
    [0.50, "#ff7f0e"], [0.74, "#ff7f0e"],
    [0.75, "#2ca02c"], [1.00, "#2ca02c"],
]

fig_timeline = go.Figure(
    data=go.Heatmap(
        z=z_state,
        x=state_matrix.columns,
        y=y_labels,
        customdata=state_matrix.to_numpy(dtype=object),
        colorscale=state_colorscale,
        zmin=0,
        zmax=3,
        colorbar=dict(
            title="Encoding",
            tickvals=[0, 1, 2, 3],
            ticktext=["none", "cue", "choice", "reward_sound"],
        ),
        hovertemplate="unit=%{y}<br>session=%{x}<br>encoding=%{customdata}<extra></extra>",
    )
)

fig_timeline.update_yaxes(tickfont=dict(size=8))
fig_timeline.update_layout(
    template="plotly_white",
    title="Encoding timeline per neuron (sorted by changing/stable and brain region)",
    xaxis_title="Session",
    yaxis_title="Neuron | Region | Stability",
    height=max(500, 13 * len(y_labels)),
)
fig_timeline.update_xaxes(tickangle=-35)
fig_timeline.show()

# Evidence heatmap (continuous strength over sessions per neuron)
# strength = (obs - thr99) / |thr99| for the strongest feature in each unit-session.
ev_df = dfc[["session_id", "unit"]].copy()
strength_cols = []
for f in features:
    col = f"strength_{f}"
    ev_df[col] = (dfc[f"obs_{f}"] - dfc[f"thr99_{f}"]) / (np.abs(dfc[f"thr99_{f}"]) + 1e-12)
    strength_cols.append(col)

arr = ev_df[strength_cols].to_numpy(dtype=float)
arr_filled = np.where(np.isfinite(arr), arr, -np.inf)
has_any = np.isfinite(arr).any(axis=1)
best_idx = np.argmax(arr_filled, axis=1)
ev_df["best_feature"] = np.where(has_any, np.array(features, dtype=object)[best_idx], "none")
ev_df["best_evidence"] = np.where(has_any, arr[np.arange(len(ev_df)), best_idx], np.nan)
ev_df = ev_df.merge(
    certainty_df[["session_id", "unit", "pref_p", "is_significant_pref"]],
    on=["session_id", "unit"],
    how="left",
)

ev_matrix = (
    ev_df.pivot(index="unit", columns="session_id", values="best_evidence")
    .reindex(index=unit_order, columns=ordered_sessions)
)
feat_matrix = (
    ev_df.pivot(index="unit", columns="session_id", values="best_feature")
    .reindex(index=unit_order, columns=ordered_sessions)
    .fillna("none")
)
sig_matrix = (
    ev_df.pivot(index="unit", columns="session_id", values="is_significant_pref")
    .reindex(index=unit_order, columns=ordered_sessions)
    .fillna(False)
    .astype(bool)
)

finite_vals = ev_matrix.to_numpy(dtype=float)
finite_vals = finite_vals[np.isfinite(finite_vals)]
if finite_vals.size:
    vmax = float(np.nanpercentile(np.abs(finite_vals), 95))
    vmax = max(vmax, 0.1)
else:
    vmax = 1.0

custom_evidence = np.dstack([
    feat_matrix.to_numpy(dtype=object),
    sig_matrix.to_numpy(dtype=object),
])

fig_evidence = go.Figure(
    data=go.Heatmap(
        z=ev_matrix.to_numpy(dtype=float),
        x=ev_matrix.columns,
        y=y_labels,
        customdata=custom_evidence,
        colorscale="RdBu_r",
        zmin=-vmax,
        zmax=vmax,
        zmid=0,
        colorbar=dict(title="Best evidence (norm. obs-thr99)"),
        hovertemplate=(
            "unit=%{y}<br>session=%{x}<br>best_feature=%{customdata[0]}"
            "<br>is_significant=%{customdata[1]}<br>evidence=%{z:.3f}<extra></extra>"
        ),
    )
)
fig_evidence.update_yaxes(tickfont=dict(size=8))
fig_evidence.update_layout(
    template="plotly_white",
    title="Encoding evidence heatmap per neuron over sessions",
    xaxis_title="Session",
    yaxis_title="Neuron | Region | Stability",
    height=max(500, 13 * len(y_labels)),
)
fig_evidence.update_xaxes(tickangle=-35)
fig_evidence.show()

# Multi-encoding longitudinal analysis (broad event classes)
multi_flag_cols = {
    "cue": "is_cue_cell",
    "choice": "is_choice_cell",
    "reward_sound": "is_reward_sound_cell",
}

multi_timeline = row_df[
    ["unit", "session_id", "session_order", "brain_region", *multi_flag_cols.values()]
].drop_duplicates(subset=["unit", "session_id"]).copy()

for col in multi_flag_cols.values():
    multi_timeline[col] = multi_timeline[col].fillna(False).astype(bool)


def _combo_from_flags(r: pd.Series) -> str:
    labels = [name for name, col in multi_flag_cols.items() if bool(r[col])]
    return "+".join(labels) if labels else "none"


multi_timeline["multi_encoding_label"] = multi_timeline.apply(_combo_from_flags, axis=1)
multi_timeline["n_zone_encodings"] = (
    multi_timeline[list(multi_flag_cols.values())].sum(axis=1).astype(int)
)
multi_timeline["is_multi_encoding"] = multi_timeline["n_zone_encodings"] >= 2


def _count_transitions(values) -> int:
    vals = list(values)
    return int(sum(a != b for a, b in zip(vals[:-1], vals[1:])))


def _multi_unit_summary(g: pd.DataFrame) -> pd.Series:
    if "session_order" in g.columns and g["session_order"].notna().any():
        g_sorted = g.sort_values(["session_order", "session_id"])
    else:
        g_sorted = g.sort_values("session_id")

    n_total = int(g_sorted["session_id"].nunique())
    n_multi = int(g_sorted["is_multi_encoding"].sum())
    n_any = int((g_sorted["n_zone_encodings"] >= 1).sum())
    multi_only = g_sorted[g_sorted["is_multi_encoding"]]

    return pd.Series(
        {
            "n_sessions_total": n_total,
            "n_any_encoding_sessions": n_any,
            "n_multi_sessions": n_multi,
            "frac_multi_sessions": (n_multi / n_total) if n_total else np.nan,
            "ever_multi_encoded": bool(n_multi > 0),
            "n_unique_multi_labels": int(multi_only["multi_encoding_label"].nunique()) if n_multi else 0,
            "label_transitions_all_sessions": _count_transitions(g_sorted["multi_encoding_label"].tolist()),
            "label_transitions_multi_sessions": _count_transitions(multi_only["multi_encoding_label"].tolist()),
        }
    )


multi_unit_profile = multi_timeline.groupby("unit").apply(_multi_unit_summary).reset_index()
multi_unit_profile = unit_profile.merge(multi_unit_profile, on="unit", how="left")
multi_unit_profile["ever_multi_encoded"] = multi_unit_profile["ever_multi_encoded"].fillna(False).astype(bool)
for col in [
    "n_sessions_total",
    "n_any_encoding_sessions",
    "n_multi_sessions",
    "n_unique_multi_labels",
    "label_transitions_all_sessions",
    "label_transitions_multi_sessions",
]:
    if col in multi_unit_profile.columns:
        multi_unit_profile[col] = multi_unit_profile[col].fillna(0).astype(int)
multi_unit_profile["frac_multi_sessions"] = multi_unit_profile["frac_multi_sessions"].fillna(0.0)

n_total_units = int(multi_unit_profile["unit"].nunique())
n_ever_multi = int(multi_unit_profile["ever_multi_encoded"].sum())
print(
    f"Units with multi-zone encoding in at least one session: {n_ever_multi}/{n_total_units} "
    f"({100 * n_ever_multi / max(n_total_units, 1):.1f}%)"
)

session_multi = (
    multi_timeline.groupby(["session_id", "session_order"], as_index=False)
    .agg(
        n_units=("unit", "nunique"),
        n_multi_units=("is_multi_encoding", "sum"),
        mean_n_zone_encodings=("n_zone_encodings", "mean"),
    )
    .sort_values(["session_order", "session_id"])
)
session_multi["pct_multi_units"] = 100.0 * session_multi["n_multi_units"] / session_multi["n_units"].clip(lower=1)

fig_multi_dynamics = px.line(
    session_multi,
    x="session_id",
    y="pct_multi_units",
    markers=True,
    title="Multi-encoding prevalence across sessions (% neurons with >=2 encodings)",
    labels={
        "session_id": "Session",
        "pct_multi_units": "% of neurons with >=2 encodings",
    },
)
fig_multi_dynamics.update_layout(template="plotly_white")
fig_multi_dynamics.update_yaxes(range=[0, max(5, min(100, session_multi["pct_multi_units"].max() * 1.15))])
fig_multi_dynamics.update_xaxes(tickangle=-35)
fig_multi_dynamics.show()

if n_ever_multi == 0:
    print("Skipping multi-encoding timeline heatmap because no neurons showed >=2 simultaneous encodings.")
else:
    multi_plot_profile = multi_unit_profile[multi_unit_profile["ever_multi_encoded"]].copy()
    multi_plot_profile["_stab_rank"] = multi_plot_profile["stability_class"].map(stab_rank).fillna(99)
    multi_plot_profile = multi_plot_profile.sort_values(
        ["_stab_rank", "brain_region", "n_multi_sessions", "unit"],
        ascending=[True, True, False, True],
    )
    multi_unit_order = multi_plot_profile["unit"].tolist()

    combo_matrix = (
        multi_timeline.pivot(index="unit", columns="session_id", values="multi_encoding_label")
        .reindex(index=multi_unit_order, columns=ordered_sessions)
        .fillna("none")
    )
    nzone_matrix = (
        multi_timeline.pivot(index="unit", columns="session_id", values="n_zone_encodings")
        .reindex(index=multi_unit_order, columns=ordered_sessions)
        .fillna(0)
        .astype(int)
    )

    combo_order = [
        "none",
        "cue",
        "choice",
        "reward_sound",
        "cue+choice",
        "cue+reward_sound",
        "choice+reward_sound",
        "cue+choice+reward_sound",
    ]
    combo_colors = {
        "none": "#d9d9d9",
        "cue": "#1f77b4",
        "choice": "#ff7f0e",
        "reward_sound": "#2ca02c",
        "cue+choice": "#9467bd",
        "cue+reward_sound": "#17becf",
        "choice+reward_sound": "#bcbd22",
        "cue+choice+reward_sound": "#d62728",
    }

    def _discrete_colorscale(colors):
        n = len(colors)
        if n == 1:
            return [[0.0, colors[0]], [1.0, colors[0]]]
        cs = []
        for i, color in enumerate(colors):
            cs.append([i / n, color])
            cs.append([(i + 1) / n, color])
        return cs

    combo_to_z = {label: i + 0.5 for i, label in enumerate(combo_order)}
    z_combo = combo_matrix.replace(combo_to_z).to_numpy(dtype=float)

    multi_label_map = multi_plot_profile.set_index("unit").apply(
        lambda r: (
            f"{r.name} | {r['brain_region']} | {r['stability_class']} | "
            f"multi={int(r['n_multi_sessions'])}"
        ),
        axis=1,
    )
    y_multi = [multi_label_map.get(u, u) for u in combo_matrix.index]

    customdata = np.dstack([
        combo_matrix.to_numpy(dtype=object),
        nzone_matrix.to_numpy(dtype=object),
    ])

    fig_multi_timeline = go.Figure(
        data=go.Heatmap(
            z=z_combo,
            x=combo_matrix.columns,
            y=y_multi,
            customdata=customdata,
            colorscale=_discrete_colorscale([combo_colors[k] for k in combo_order]),
            zmin=0,
            zmax=len(combo_order),
            colorbar=dict(
                title="Encoding combo",
                tickvals=[i + 0.5 for i in range(len(combo_order))],
                ticktext=combo_order,
            ),
            hovertemplate=(
                "unit=%{y}<br>session=%{x}<br>encoding_combo=%{customdata[0]}"
                "<br>n_zone_encodings=%{customdata[1]}<extra></extra>"
            ),
        )
    )

    fig_multi_timeline.update_yaxes(tickfont=dict(size=8))
    fig_multi_timeline.update_layout(
        template="plotly_white",
        title=(
            "Multi-encoding timeline per neuron (ever multi-encoded; sorted by changing/stable and brain region)"
        ),
        xaxis_title="Session",
        yaxis_title="Neuron | Region | Stability | # multi sessions",
        height=max(550, 14 * len(y_multi)),
    )
    fig_multi_timeline.update_xaxes(tickangle=-35)
    fig_multi_timeline.show()

# tables iof data
stable_units = unit_profile[unit_profile["stability_class"] == "stable"].sort_values(["brain_region", "unit"])
changing_units = unit_profile[unit_profile["stability_class"] == "changing"].sort_values(["brain_region", "unit"])

display(stable_units.head(20))
display(changing_units.head(20))




2026-03-06 12:42:29,537|DEBUG|2193354|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-03-06 12:42:29,539|DEBUG|2193354|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-03-06 12:42:29,540|DEBUG|2193354|analytics|get_analytics
	Processing SpikeClusterMetadata, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/mnt/slow/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-03-06 12:42:29,540|INFO|2193354|analytics|get_analytics
	Analytic `SpikeClusterMetadata` does not exist for (1100, 6, '2024-11-14_15-01'), compute first, or check for typo
2026-03-06 12:42:29,541|DEBUG|2193354|analytics|get_analytics
	Processing SpikeClusterMetadata, (1100, 6, '2024-11-14_16-40') 2024-11-14_16-40_rYL006_P1100_LinearTra

2026-03-06 12:42:30,205|DEBUG|2193354|analytics|get_analytics
	Processing SpikeClusterMetadata, (1100, 6, '2024-11-27_16-11') 2024-11-27_16-11_rYL006_P1100_LinearTrackStop_31min.hdf5
/mnt/slow/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-27_16-11_rYL006_P1100_LinearTrackStop_31min
2026-03-06 12:42:30,206|INFO|2193354|analytics|get_analytics
	Analytic `SpikeClusterMetadata` does not exist for (1100, 6, '2024-11-27_16-11'), compute first, or check for typo
2026-03-06 12:42:30,207|DEBUG|2193354|analytics|get_analytics
	Processing SpikeClusterMetadata, (1100, 6, '2024-11-28_17-41') 2024-11-28_17-41_rYL006_P1100_LinearTrackStop_30min.hdf5
/mnt/slow/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-28_17-41_rYL006_P1100_LinearTrackStop_30min
2026-03-06 12:42:30,306|DEBUG|2193354|analytics|get_analytics
	Processing SpikeClusterMetadata, (1100, 6, '2024-12-02_16-09') 2024-12-02_16-09_rYL006_P1100_LinearTrackStop_28min.hdf5
/mnt/slow/BMI/Vi

Units total: 77 | stable: 16 | changing: 54 | unclassified: 2


/tmp/ipykernel_2193354/2736572332.py:234: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



/tmp/ipykernel_2193354/2736572332.py:412: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



Units with multi-zone encoding in at least one session: 42/77 (54.5%)


/tmp/ipykernel_2193354/2736572332.py:516: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



,unit,brain_region,dominant_encoding,stability_class,n_sessions,n_sig_sessions,n_unique_sig_encodings
1,Unit0002,CA1,choice,stable,25,10,1
3,Unit0004,DG,choice,stable,25,3,1
7,Unit0008,DG,cue,stable,25,10,1
11,Unit0012,DG,choice,stable,25,6,1
12,Unit0013,DG,choice,stable,25,3,1
14,Unit0015,DG,cue,stable,25,24,1
16,Unit0017,DG,reward_sound,stable,25,9,1
17,Unit0018,DG,reward_sound,stable,25,8,1
76,Unit0077,InfrL,reward_sound,stable,25,3,1
36,Unit0037,PrL,reward_sound,stable,25,5,1


,unit,brain_region,dominant_encoding,stability_class,n_sessions,n_sig_sessions,n_unique_sig_encodings
20,Unit0021,ACC,choice,changing,25,11,2
21,Unit0022,ACC,choice,changing,25,14,2
22,Unit0023,ACC,choice,changing,25,5,3
23,Unit0024,ACC,choice,changing,25,23,2
24,Unit0025,ACC,choice,changing,25,15,3
25,Unit0026,ACC,choice,changing,25,10,2
26,Unit0027,ACC,reward_sound,changing,25,17,2
0,Unit0001,CA1,choice,changing,25,9,2
4,Unit0005,DG,reward_sound,changing,25,4,2
5,Unit0006,DG,reward_sound,changing,25,12,2


In [9]:
t0_ens

from_ephys_timestamp  \
paradigm_id animal_id session_id       entry_id                         
1100        6         2024-11-14_16-40 0                      4400000   
                                       1                      4440000   
                                       2                      4480000   
                                       3                      4520000   
                                       4                      4560000   
...                                                               ...   
                      2025-01-27_13-39 39795               4275440000   
                                       39796               4275480000   
                                       39797               4275520000   
                                       39798               4275560000   
                                       39799               4275600000   

                                                 to_ephys_timestamp  \
paradigm_id animal_id session_id       entry_id                       
1100        6         2024-11-14_16-40 0                    4440000   
                                       1                    4480000   
                                       2                    4520000   
                                       3                    4560000   
                                       4                    4600000   
...                                                             ...   
                      2025-01-27_13-39 39795             4275480000   
                                       39796             4275520000   
                                       39797             4275560000   
                                       39798             4275600000   
                                       39799             4275640000   

                                                 Assembly001  Assembly002  \
paradigm_id animal_id session_id       entry_id                             
1100        6         2024-11-14_16-40 0            0.044591    -0.079732   
                                       1            0.353522    -0.208656   
                                       2           -0.090020    -0.381483   
                                       3           -0.082682    -0.003392   
                                       4            0.169519    -0.213003   
...                                                      ...          ...   
                      2025-01-27_13-39 39795       -0.121557    -0.456305   
                                       39796       -0.166517    -0.094205   
                                       39797       -0.062079    -0.142495   
                                       39798       -0.032098     0.095632   
                                       39799        0.065874     0.164919   

                                                 Assembly003  Assembly004  \
paradigm_id animal_id session_id       entry_id                             
1100        6         2024-11-14_16-40 0            0.117157    -0.100619   
                                       1           -0.050525    -0.098752   
                                       2            0.185447     1.103287   
                                       3            0.093962    -0.152654   
                                       4           -0.009496    -0.107002   
...                                                      ...          ...   
                      2025-01-27_13-39 39795        1.920625    -1.375872   
                                       39796        0.436531    -0.603980   
                                       39797       -0.051984    -0.121114   
                                       39798       -0.241508     0.670083   
                                       39799       -0.220040     0.440637   

                                                 Assembly005  Assembly006  \
paradigm_id animal_id session_id       entry_id                             
1100        6    

In [14]:
# Time-resolved decoding on ensemble projections (higher-sample version)
# Key updates:
# 1) Rebuild interval-relative time from (session_id, trial_id, interval_name).
# 2) Pool neighboring relative bins to increase per-timepoint sample count.
# 3) Run CV with n_jobs=1 to avoid multiprocessing resource_tracker errors in notebooks.
# 4) Use binary outcome derived from cue+choice and binary cue classes (1 vs 2).

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load ensemble trialwise/timestamp projection if not already present
if 't0_ens' not in globals() or t0_ens is None:
    t0_ens = analytics.get_analytics('EnsembleT0Projection', session_names=session_names)

ens_df = t0_ens.copy()
if isinstance(ens_df.index, pd.MultiIndex) or ens_df.index.name is not None:
    ens_df = ens_df.reset_index()


ASSEMBLY_MODE = 'single'  # 'single' or 'all'
assembly_col = 'Assembly012'

N_REL_BINS = 20              # fixed bins from start->end of each interval
WINDOW_HALF_WIDTH = 1        # use neighboring rel bins [+/- this many] for each classifier
MIN_SAMPLES_PER_TIME = 20
MIN_CLASS_COUNT = 4
RANDOM_STATE = 42

USE_GROUP_CV = True          # True: split by session where possible
N_JOBS = 1                   # keep at 1 to avoid ChildProcessError/resource_tracker noise

# Outcome rule (from your specification):
# cue==1 and choice_R1 positive -> True
# cue==2 and choice_R2 positive -> True
# choice_R2 appears to be 0/1 in this dataset; keep {1,2} for robustness.
OUTCOME_R1_POSITIVE_VALUES = {1}
OUTCOME_R2_POSITIVE_VALUES = {1, 2}

interval_col = 'interval_name' if 'interval_name' in ens_df.columns else None
if interval_col is None:
    raise ValueError('Expected column interval_name in t0_ens for interval-wise decoding.')

session_col = 'session_id' if 'session_id' in ens_df.columns else None
if session_col is None:
    raise ValueError('Expected session_id in t0_ens (from index after reset_index).')

trial_col = next((c for c in ['trial_id', 'behavior_trial_id', 'trial', 'trial_index'] if c in ens_df.columns), None)
if trial_col is None:
    raise ValueError('Expected a trial column (trial_id/behavior_trial_id/trial/trial_index).')

assembly_cols = [c for c in ens_df.columns if str(c).startswith('Assembly')]
if ASSEMBLY_MODE == 'single':
    if assembly_col not in ens_df.columns:
        raise ValueError(
            f"{assembly_col} not found. Available Assembly columns: "
            f"{assembly_cols[:10]}{'...' if len(assembly_cols) > 10 else ''}"
        )
    feature_cols = [assembly_col]
else:
    if len(assembly_cols) == 0:
        raise ValueError('No Assembly* columns found in t0_ens.')
    feature_cols = assembly_cols

print(f'Using {len(feature_cols)} feature(s): {feature_cols[:8]}{" ..." if len(feature_cols) > 8 else ""}')

# Build interval-relative time by enumerating bins inside each session/trial/interval block.
sort_cols = [session_col, trial_col, interval_col]
for c in ['from_ephys_timestamp', 'to_ephys_timestamp', 'entry_id']:
    if c in ens_df.columns:
        sort_cols.append(c)
ens_df = ens_df.sort_values(sort_cols).copy()

ens_df['time_bin_idx'] = (
    ens_df.groupby([session_col, trial_col, interval_col], sort=False, dropna=False)
    .cumcount()
    .astype(int)
)

ens_df['n_bins_interval'] = (
    ens_df.groupby([session_col, trial_col, interval_col], sort=False, dropna=False)['time_bin_idx']
    .transform('max')
    .add(1)
    .astype(int)
)

# Relative position in interval [0,1]
ens_df['time_rel'] = np.where(
    ens_df['n_bins_interval'] > 1,
    ens_df['time_bin_idx'] / (ens_df['n_bins_interval'] - 1),
    0.0,
)

ens_df['time_rel_bin'] = np.clip(
    np.round(ens_df['time_rel'] * (N_REL_BINS - 1)).astype(int),
    0,
    N_REL_BINS - 1,
)

# Infer bin width in seconds for optional sec-axis labels
def _infer_bin_width_s(df):
    if {'from_ephys_timestamp', 'to_ephys_timestamp'}.issubset(df.columns):
        width = pd.to_numeric(df['to_ephys_timestamp'], errors='coerce') - pd.to_numeric(df['from_ephys_timestamp'], errors='coerce')
        width = width.dropna()
        if not width.empty:
            return float(width.median()) / 1e6
    return 0.04

bin_width_s = _infer_bin_width_s(ens_df)
# rough sec mapping per interval using median interval length in bins
interval_len_bins = (
    ens_df.groupby(interval_col, dropna=False)['n_bins_interval']
    .median()
    .to_dict()
)

# Numeric cleanup
for col in feature_cols:
    ens_df[col] = pd.to_numeric(ens_df[col], errors='coerce')
for tgt in ['trial_outcome', 'choice_R1', 'choice_R2', 'cue']:
    if tgt in ens_df.columns:
        ens_df[tgt] = pd.to_numeric(ens_df[tgt], errors='coerce')

# Build binary outcome from cue+choice, per your requested rule
required_for_outcome = {'cue', 'choice_R1', 'choice_R2'}
if required_for_outcome.issubset(ens_df.columns):
    ens_df['outcome_binary'] = (
        ((ens_df['cue'] == 1) & (ens_df['choice_R1'].isin(OUTCOME_R1_POSITIVE_VALUES)))
        | ((ens_df['cue'] == 2) & (ens_df['choice_R2'].isin(OUTCOME_R2_POSITIVE_VALUES)))
    ).astype(float)
else:
    ens_df['outcome_binary'] = np.nan

# Match requested targets
targets = {
    'Outcome': 'outcome_binary',
    'Cue': 'cue',
    'R1 choice': 'choice_R1',
    'R2 choice': 'choice_R2',
}


def _build_pipeline():
    return Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=2000, class_weight='balanced')),
        ]
    )


def _evaluate_time_slice(slice_df, target_col):
    X = slice_df[feature_cols].to_numpy()
    y = slice_df[target_col].to_numpy()

    cls_counts = pd.Series(y).value_counts(dropna=True)
    if len(cls_counts) < 2:
        return np.nan, np.nan, 0, 'single_class'
    if cls_counts.min() < MIN_CLASS_COUNT:
        return np.nan, np.nan, 0, 'too_few_per_class'

    model = _build_pipeline()

    if USE_GROUP_CV:
        g = slice_df[session_col].astype(str)
        if g.nunique() >= 3:
            n_splits = int(min(5, g.nunique()))
            if n_splits >= 2:
                try:
                    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
                    out = cross_validate(
                        model,
                        X,
                        y,
                        groups=g,
                        cv=cv,
                        scoring={'acc': 'accuracy', 'bal_acc': 'balanced_accuracy'},
                        n_jobs=N_JOBS,
                        error_score=np.nan,
                    )
                    return float(np.nanmean(out['test_acc'])), float(np.nanmean(out['test_bal_acc'])), n_splits, 'stratified_group'
                except ValueError:
                    pass

    n_splits = int(min(5, cls_counts.min()))
    if n_splits < 2:
        return np.nan, np.nan, 0, 'cv_not_possible'

    try:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
        out = cross_validate(
            model,
            X,
            y,
            cv=cv,
            scoring={'acc': 'accuracy', 'bal_acc': 'balanced_accuracy'},
            n_jobs=N_JOBS,
            error_score=np.nan,
        )
        return float(np.nanmean(out['test_acc'])), float(np.nanmean(out['test_bal_acc'])), n_splits, 'stratified'
    except ValueError:
        return np.nan, np.nan, 0, 'cv_error'


rows = []

for target_name, target_col in targets.items():
    if target_col not in ens_df.columns:
        continue

    cols = [session_col, trial_col, interval_col, 'time_rel', 'time_rel_bin'] + feature_cols + [target_col]
    df_t = ens_df[cols].copy()
    df_t = df_t.dropna(subset=[session_col, trial_col, interval_col, 'time_rel', 'time_rel_bin', target_col] + feature_cols)

    df_t[target_col] = pd.to_numeric(df_t[target_col], errors='coerce')
    df_t = df_t.dropna(subset=[target_col])

    # Enforce binary classes where needed
    if target_col == 'cue':
        # Exclude rare cue=0 rows, keep Cue1 vs Cue2 only
        df_t = df_t[df_t[target_col].isin([1, 2])]
    elif target_col in ['choice_R1', 'choice_R2', 'outcome_binary']:
        df_t = df_t[df_t[target_col].isin([0, 1])]

    df_t[target_col] = df_t[target_col].astype(int)

    if df_t.empty:
        continue

    # One sample per trial per rel-bin (mean over raw bins that map to that rel-bin)
    agg_map = {c: 'mean' for c in feature_cols}
    agg_map['time_rel'] = 'mean'
    agg_map[target_col] = 'first'

    trial_bin_df = (
        df_t.groupby([session_col, trial_col, interval_col, 'time_rel_bin'], as_index=False)
        .agg(agg_map)
    )

    for interval_name, interval_df in trial_bin_df.groupby(interval_col):
        interval_bins = sorted(interval_df['time_rel_bin'].unique())
        for b in interval_bins:
            # Pool neighboring bins to increase sample count
            lo = b - WINDOW_HALF_WIDTH
            hi = b + WINDOW_HALF_WIDTH
            win = interval_df[interval_df['time_rel_bin'].between(lo, hi)].copy()

            if win.empty:
                continue

            # Keep one sample per trial after window pooling
            per_trial = (
                win.groupby([session_col, trial_col], as_index=False)
                .agg({**{c: 'mean' for c in feature_cols}, target_col: 'first', 'time_rel': 'mean'})
            )

            n_samples = len(per_trial)
            if n_samples < MIN_SAMPLES_PER_TIME:
                rows.append(
                    {
                        'feature_mode': ASSEMBLY_MODE,
                        'assembly': assembly_col if ASSEMBLY_MODE == 'single' else 'ALL',
                        'n_features': len(feature_cols),
                        'target_name': target_name,
                        'target_col': target_col,
                        'interval_name': str(interval_name),
                        'time_rel_bin': int(b),
                        'time_rel': float(per_trial['time_rel'].median()),
                        'time_s': np.nan,
                        'n_samples': int(n_samples),
                        'accuracy': np.nan,
                        'balanced_accuracy': np.nan,
                        'n_splits': 0,
                        'cv_strategy': 'too_few_samples',
                    }
                )
                continue

            acc, bacc, n_splits, cv_name = _evaluate_time_slice(per_trial, target_col)

            # Approximate sec scale for this interval
            med_n = float(interval_len_bins.get(interval_name, np.nan))
            if np.isnan(med_n) or med_n <= 1:
                t_s = np.nan
            else:
                t_s = float(b / (N_REL_BINS - 1) * (med_n - 1) * bin_width_s)

            rows.append(
                {
                    'feature_mode': ASSEMBLY_MODE,
                    'assembly': assembly_col if ASSEMBLY_MODE == 'single' else 'ALL',
                    'n_features': len(feature_cols),
                    'target_name': target_name,
                    'target_col': target_col,
                    'interval_name': str(interval_name),
                    'time_rel_bin': int(b),
                    'time_rel': float(per_trial['time_rel'].median()),
                    'time_s': t_s,
                    'n_samples': int(n_samples),
                    'accuracy': acc,
                    'balanced_accuracy': bacc,
                    'n_splits': int(n_splits),
                    'cv_strategy': cv_name,
                }
            )

decode_time_resolved = pd.DataFrame(rows)
print(f"Rows in decode_time_resolved: {len(decode_time_resolved)}")
print(f"bin_width_s={bin_width_s:.6f} s | N_REL_BINS={N_REL_BINS} | WINDOW_HALF_WIDTH={WINDOW_HALF_WIDTH}")

if not decode_time_resolved.empty:
    print()
    print('CV strategy counts:')
    display(decode_time_resolved['cv_strategy'].value_counts(dropna=False).to_frame('n_rows'))
    print()
    print('Per-target sample summary (all intervals/time bins):')
    display(decode_time_resolved.groupby('target_name')['n_samples'].describe())
    print()
    print('Per-target class coverage in valid rows:')
    valid = decode_time_resolved[decode_time_resolved['accuracy'].notna()]
    display(valid.groupby('target_name').size().to_frame('n_valid_rows'))

display(decode_time_resolved.head())

for target_name in decode_time_resolved['target_name'].dropna().unique():
    sub = decode_time_resolved[
        (decode_time_resolved['target_name'] == target_name)
        & decode_time_resolved['accuracy'].notna()
    ].sort_values(['interval_name', 'time_rel_bin'])

    if sub.empty:
        print(f'No valid CV results for target: {target_name}')
        continue

    fig = px.line(
        sub,
        x='time_rel',
        y='accuracy',
        color='interval_name',
        markers=True,
        title=f"{sub['assembly'].iloc[0]} decoding over relative interval time: {target_name}",
        labels={
            'time_rel': 'Relative time in interval (0=start, 1=end)',
            'accuracy': 'CV accuracy',
            'interval_name': 'Interval',
        },
    )
    fig.update_layout(template='plotly_white')
    fig.show()

for target_name in decode_time_resolved['target_name'].dropna().unique():
    sub = decode_time_resolved[
        (decode_time_resolved['target_name'] == target_name)
        & decode_time_resolved['balanced_accuracy'].notna()
    ].sort_values(['interval_name', 'time_rel_bin'])

    if sub.empty:
        continue

    fig = px.line(
        sub,
        x='time_rel',
        y='balanced_accuracy',
        color='interval_name',
        markers=True,
        title=f"{sub['assembly'].iloc[0]} balanced accuracy over relative interval time: {target_name}",
        labels={
            'time_rel': 'Relative time in interval (0=start, 1=end)',
            'balanced_accuracy': 'CV balanced accuracy',
            'interval_name': 'Interval',
        },
    )
    fig.update_layout(template='plotly_white')
    fig.show()



Using 1 feature(s): ['Assembly012']
Rows in decode_time_resolved: 680
bin_width_s=0.040000 s | N_REL_BINS=20 | WINDOW_HALF_WIDTH=1

CV strategy counts:


,n_rows
cv_strategy,
stratified_group,560
single_class,120



Per-target sample summary (all intervals/time bins):


,count,mean,std,min,25%,50%,75%,max
target_name,,,,,,,,
Cue,170.0,2158.176471,1003.239618,332.0,2713.0,2713.0,2713.0,2713.0
Outcome,170.0,2160.470588,1004.515846,332.0,2716.0,2716.0,2716.0,2716.0
R1 choice,170.0,2160.470588,1004.515846,332.0,2716.0,2716.0,2716.0,2716.0
R2 choice,170.0,2160.470588,1004.515846,332.0,2716.0,2716.0,2716.0,2716.0



Per-target class coverage in valid rows:


,n_valid_rows
target_name,
Cue,130
Outcome,130
R1 choice,150
R2 choice,150


,feature_mode,assembly,n_features,target_name,target_col,interval_name,time_rel_bin,time_rel,time_s,n_samples,accuracy,balanced_accuracy,n_splits,cv_strategy
0,single,Assembly012,1,Outcome,outcome_binary,R1_entry_interval,0,0.038462,0.000000,2716,0.491661,0.490897,5,stratified_group
1,single,Assembly012,1,Outcome,outcome_binary,R1_entry_interval,1,0.064103,0.082105,2716,0.461835,0.480937,5,stratified_group
2,single,Assembly012,1,Outcome,outcome_binary,R1_entry_interval,2,0.115385,0.164211,2716,0.502076,0.501938,5,stratified_group
3,single,Assembly012,1,Outcome,outcome_binary,R1_entry_interval,3,0.166667,0.246316,2716,0.500418,0.500866,5,stratified_group
4,single,Assembly012,1,Outcome,outcome_binary,R1_entry_interval,4,0.217949,0.328421,2716,0.497624,0.496891,5,stratified_group


In [15]:
# Leave-one-trial-out decoding (fresh cell)
# This cell implements:
# 1) Interval exclusion: pre_cue_interval + nextto_cue_interval (+ typo variant)
# 2) Overlapping relative-time bins with explicit sample-count diagnostics
# 3) Leave-one-trial-out evaluation at each interval x time bin

from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# Settings
ASSEMBLY_MODE = 'single'      # 'single' or 'all'
ASSEMBLY_COL = 'Assembly012'  # used if ASSEMBLY_MODE='single'

N_REL_BINS = 20               # relative bins inside each interval
WINDOW_HALF_WIDTH = 1         # overlap: use bins [b-1, b, b+1]

MIN_SAMPLES_PER_BIN = 20
MIN_CLASS_COUNT = 3

# Requested interval exclusions
EXCLUDED_INTERVALS = {
    'pre_cue_interval',
    'pre_cue_itnerval',  # keep typo variant explicitly
    'nextto_cue_interval',
}

# Outcome rule requested by you
OUTCOME_R1_POSITIVE_VALUES = {1}
OUTCOME_R2_POSITIVE_VALUES = {1, 2}  # dataset appears to use 0/1, keep 2 for robustness

# If needed, load t0_ens
if 't0_ens' not in globals() or t0_ens is None:
    t0_ens = analytics.get_analytics('EnsembleT0Projection', session_names=session_names)

ens_df = t0_ens.copy()
if isinstance(ens_df.index, pd.MultiIndex) or ens_df.index.name is not None:
    ens_df = ens_df.reset_index()

# Required columns
session_col = 'session_id'
interval_col = 'interval_name'
trial_col = next((c for c in ['trial_id', 'behavior_trial_id', 'trial', 'trial_index'] if c in ens_df.columns), None)
if trial_col is None:
    raise ValueError('No trial column found (expected trial_id/behavior_trial_id/trial/trial_index).')
if interval_col not in ens_df.columns:
    raise ValueError('interval_name column not found in t0_ens.')
if session_col not in ens_df.columns:
    raise ValueError('session_id column not found in t0_ens.')

# Exclude requested intervals
ens_df = ens_df[~ens_df[interval_col].astype(str).isin(EXCLUDED_INTERVALS)].copy()
print('Intervals used:', sorted(ens_df[interval_col].dropna().astype(str).unique().tolist()))

# Features
assembly_cols = [c for c in ens_df.columns if str(c).startswith('Assembly')]
if ASSEMBLY_MODE == 'single':
    if ASSEMBLY_COL not in ens_df.columns:
        raise ValueError(f'{ASSEMBLY_COL} not found in t0_ens.')
    feature_cols = [ASSEMBLY_COL]
else:
    if len(assembly_cols) == 0:
        raise ValueError('No Assembly* columns found in t0_ens.')
    feature_cols = assembly_cols

print(f'Feature mode: {ASSEMBLY_MODE} | n_features={len(feature_cols)}')

# Numeric cleanup
for c in feature_cols + ['cue', 'choice_R1', 'choice_R2']:
    if c in ens_df.columns:
        ens_df[c] = pd.to_numeric(ens_df[c], errors='coerce')

# Build requested outcome label
ens_df['outcome_binary'] = (
    ((ens_df['cue'] == 1) & (ens_df['choice_R1'].isin(OUTCOME_R1_POSITIVE_VALUES)))
    | ((ens_df['cue'] == 2) & (ens_df['choice_R2'].isin(OUTCOME_R2_POSITIVE_VALUES)))
).astype(float)

# Sort then build interval-relative bins
sort_cols = [session_col, trial_col, interval_col]
for c in ['from_ephys_timestamp', 'to_ephys_timestamp', 'entry_id']:
    if c in ens_df.columns:
        sort_cols.append(c)
ens_df = ens_df.sort_values(sort_cols).copy()

ens_df['time_bin_idx'] = (
    ens_df.groupby([session_col, trial_col, interval_col], sort=False, dropna=False)
    .cumcount()
    .astype(int)
)
ens_df['n_bins_interval'] = (
    ens_df.groupby([session_col, trial_col, interval_col], sort=False, dropna=False)['time_bin_idx']
    .transform('max')
    .add(1)
    .astype(int)
)
ens_df['time_rel'] = np.where(
    ens_df['n_bins_interval'] > 1,
    ens_df['time_bin_idx'] / (ens_df['n_bins_interval'] - 1),
    0.0,
)
ens_df['time_rel_bin'] = np.clip(
    np.round(ens_df['time_rel'] * (N_REL_BINS - 1)).astype(int),
    0,
    N_REL_BINS - 1,
)

# Unique trial identity across sessions
ens_df['trial_uid'] = ens_df[session_col].astype(str) + '__' + ens_df[trial_col].astype(str)

# Targets
# Cue: decode cue 1 vs 2 only (drop rare cue==0)
# Outcome: decode outcome_binary 0 vs 1
# R1/R2 choices: decode 0 vs 1
targets = {
    'Outcome': 'outcome_binary',
    'Cue': 'cue',
    'R1 choice': 'choice_R1',
    'R2 choice': 'choice_R2',
}

def _build_model():
    return Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=1500, class_weight='balanced', solver='liblinear')),
        ]
    )

def _prepare_target_df(df, target_col):
    cols = [session_col, trial_col, 'trial_uid', interval_col, 'time_rel', 'time_rel_bin'] + feature_cols + [target_col]
    out = df[cols].copy().dropna(subset=[session_col, trial_col, 'trial_uid', interval_col, 'time_rel', 'time_rel_bin', target_col] + feature_cols)

    out[target_col] = pd.to_numeric(out[target_col], errors='coerce')
    out = out.dropna(subset=[target_col])

    # enforce binary targets
    if target_col == 'cue':
        out = out[out[target_col].isin([1, 2])]  # Cue1 vs Cue2 only
    else:
        out = out[out[target_col].isin([0, 1])]

    out[target_col] = out[target_col].astype(int)
    return out

def _trial_level_bin_agg(df_t, target_col):
    # one row per trial x relative bin
    agg_map = {c: 'mean' for c in feature_cols}
    agg_map['time_rel'] = 'mean'
    agg_map[target_col] = 'first'

    return (
        df_t.groupby([session_col, trial_col, 'trial_uid', interval_col, 'time_rel_bin'], as_index=False)
        .agg(agg_map)
    )

def _window_pool(bin_df, target_col, center_bin):
    lo = center_bin - WINDOW_HALF_WIDTH
    hi = center_bin + WINDOW_HALF_WIDTH
    w = bin_df[bin_df['time_rel_bin'].between(lo, hi)].copy()
    if w.empty:
        return w

    # one row per trial after pooling (keeps trial as unit of analysis)
    pooled = (
        w.groupby([session_col, trial_col, 'trial_uid'], as_index=False)
        .agg({**{c: 'mean' for c in feature_cols}, target_col: 'first', 'time_rel': 'mean'})
    )
    return pooled

def _fit_loto(per_trial_df, target_col):
    X = per_trial_df[feature_cols].to_numpy()
    y = per_trial_df[target_col].to_numpy().astype(int)
    groups = per_trial_df['trial_uid'].to_numpy()

    cls_counts = pd.Series(y).value_counts(dropna=True)
    if len(cls_counts) < 2:
        return np.nan, np.nan, 0, 'single_class'
    if cls_counts.min() < MIN_CLASS_COUNT:
        return np.nan, np.nan, 0, 'too_few_per_class'

    n_groups = int(pd.Series(groups).nunique())
    if n_groups < 3:
        return np.nan, np.nan, n_groups, 'too_few_trials'

    model = _build_model()
    cv = LeaveOneGroupOut()

    try:
        # trial_uid is unique per row here; this is effectively leave-one-trial-out
        y_pred = cross_val_predict(
            model,
            X,
            y,
            groups=groups,
            cv=cv,
            method='predict',
            n_jobs=1,
        )
        acc = float(accuracy_score(y, y_pred))
        bacc = float(balanced_accuracy_score(y, y_pred))
        return acc, bacc, n_groups, 'loto'
    except Exception:
        return np.nan, np.nan, n_groups, 'fit_error'

results = []
sample_diag = []

for target_name, target_col in targets.items():
    if target_col not in ens_df.columns:
        continue

    df_t = _prepare_target_df(ens_df, target_col)
    if df_t.empty:
        print(f'{target_name}: no rows after filtering')
        continue

    tb = _trial_level_bin_agg(df_t, target_col)

    for interval_name, interval_df in tb.groupby(interval_col):
        bins = sorted(interval_df['time_rel_bin'].unique())
        for b in bins:
            pooled = _window_pool(interval_df, target_col, b)
            if pooled.empty:
                continue

            n_samples = int(len(pooled))
            n_sessions = int(pooled[session_col].nunique())
            n_trials = int(pooled['trial_uid'].nunique())
            cls_counts = pooled[target_col].value_counts(dropna=True)
            min_class = int(cls_counts.min()) if len(cls_counts) else 0
            n_classes = int(len(cls_counts))

            sample_diag.append(
                {
                    'target_name': target_name,
                    'interval_name': str(interval_name),
                    'time_rel_bin': int(b),
                    'n_samples': n_samples,
                    'n_trials': n_trials,
                    'n_sessions': n_sessions,
                    'n_classes': n_classes,
                    'min_class_count': min_class,
                }
            )

            if n_samples < MIN_SAMPLES_PER_BIN:
                results.append(
                    {
                        'target_name': target_name,
                        'target_col': target_col,
                        'interval_name': str(interval_name),
                        'time_rel_bin': int(b),
                        'time_rel': float(pooled['time_rel'].median()),
                        'n_samples': n_samples,
                        'n_trials': n_trials,
                        'n_sessions': n_sessions,
                        'accuracy': np.nan,
                        'balanced_accuracy': np.nan,
                        'n_folds': 0,
                        'cv_strategy': 'too_few_samples',
                    }
                )
                continue

            acc, bacc, n_folds, cv_name = _fit_loto(pooled, target_col)
            results.append(
                {
                    'target_name': target_name,
                    'target_col': target_col,
                    'interval_name': str(interval_name),
                    'time_rel_bin': int(b),
                    'time_rel': float(pooled['time_rel'].median()),
                    'n_samples': n_samples,
                    'n_trials': n_trials,
                    'n_sessions': n_sessions,
                    'accuracy': acc,
                    'balanced_accuracy': bacc,
                    'n_folds': int(n_folds),
                    'cv_strategy': cv_name,
                }
            )

loto_decode_results = pd.DataFrame(results)
loto_sample_diagnostics = pd.DataFrame(sample_diag)

print(f'Rows in loto_decode_results: {len(loto_decode_results)}')
print(f'Rows in loto_sample_diagnostics: {len(loto_sample_diagnostics)}')

if not loto_sample_diagnostics.empty:
    print()
    print('Sample count summary per target (across interval/time bins):')
    display(loto_sample_diagnostics.groupby('target_name')['n_samples'].describe())

    print()
    print('Median n_samples per interval and target:')
    med_tbl = (
        loto_sample_diagnostics
        .groupby(['target_name', 'interval_name'], as_index=False)['n_samples']
        .median()
        .rename(columns={'n_samples': 'median_n_samples'})
        .sort_values(['target_name', 'median_n_samples'], ascending=[True, False])
    )
    display(med_tbl)

if not loto_decode_results.empty:
    print()
    print('CV strategy counts:')
    display(loto_decode_results['cv_strategy'].value_counts(dropna=False).to_frame('n_rows'))

    valid = loto_decode_results[loto_decode_results['accuracy'].notna()].copy()
    print()
    print('Valid rows per target:')
    display(valid.groupby('target_name').size().to_frame('n_valid_rows'))

    print()
    print('Mean performance per target (valid rows):')
    display(
        valid.groupby('target_name', as_index=False)
        .agg(mean_acc=('accuracy', 'mean'), mean_bal_acc=('balanced_accuracy', 'mean'))
    )

# Plot accuracy over relative interval time
for target_name in loto_decode_results['target_name'].dropna().unique():
    sub = loto_decode_results[
        (loto_decode_results['target_name'] == target_name)
        & loto_decode_results['accuracy'].notna()
    ].sort_values(['interval_name', 'time_rel_bin'])

    if sub.empty:
        print(f'No valid LOTO results for target: {target_name}')
        continue

    fig = px.line(
        sub,
        x='time_rel',
        y='accuracy',
        color='interval_name',
        markers=True,
        title=f'LOTO decoding over relative time: {target_name}',
        labels={
            'time_rel': 'Relative time in interval (0=start, 1=end)',
            'accuracy': 'LOTO accuracy',
            'interval_name': 'Interval',
        },
        hover_data=['n_samples', 'n_trials', 'n_sessions', 'n_folds'],
    )
    fig.update_layout(template='plotly_white')
    fig.show()

Intervals used: ['R1_entry_interval', 'R1_exit_interval', 'R2_entry_interval', 'R2_exit_interval', 'cue_entry_interval', 'cue_exit_interval', 'reward1_sound_interval', 'reward2_sound_interval']
Feature mode: single | n_features=1
Rows in loto_decode_results: 640
Rows in loto_sample_diagnostics: 640

Sample count summary per target (across interval/time bins):


,count,mean,std,min,25%,50%,75%,max
target_name,,,,,,,,
Cue,160.0,2123.50,1024.314712,332.0,2129.25,2713.0,2713.0,2713.0
Outcome,160.0,2125.75,1025.617746,332.0,2131.50,2716.0,2716.0,2716.0
R1 choice,160.0,2125.75,1025.617746,332.0,2131.50,2716.0,2716.0,2716.0
R2 choice,160.0,2125.75,1025.617746,332.0,2131.50,2716.0,2716.0,2716.0



Median n_samples per interval and target:


,target_name,interval_name,median_n_samples
0,Cue,R1_entry_interval,2713.0
1,Cue,R1_exit_interval,2713.0
2,Cue,R2_entry_interval,2713.0
3,Cue,R2_exit_interval,2713.0
4,Cue,cue_entry_interval,2713.0
5,Cue,cue_exit_interval,2713.0
6,Cue,reward1_sound_interval,378.0
7,Cue,reward2_sound_interval,332.0
8,Outcome,R1_entry_interval,2716.0
9,Outcome,R1_exit_interval,2716.0



CV strategy counts:


,n_rows
cv_strategy,
loto,520
single_class,120



Valid rows per target:


,n_valid_rows
target_name,
Cue,120
Outcome,120
R1 choice,140
R2 choice,140



Mean performance per target (valid rows):


,target_name,mean_acc,mean_bal_acc
0,Cue,0.524555,0.523997
1,Outcome,0.515347,0.516289
2,R1 choice,0.528188,0.515184
3,R2 choice,0.525352,0.532226


In [16]:
# Permutation significance test for time-resolved decoding
# Works with the LOTO setup from the previous cell.
# Supports two modes:
# - PERM_MODE='conditional' (fast): permutation test on out-of-fold predictions
# - PERM_MODE='refit' (strict, very slow): retrain on each permutation

from sklearn.model_selection import LeaveOneGroupOut, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score
import time

# Settings
CV_MODE = 'loto'  # 'loto' or 'session_group_kfold'
PERM_MODE = 'conditional'  # 'conditional' or 'refit'
N_PERMUTATIONS = 500
RANDOM_STATE = 42
MAX_CV_SPLITS = 5

TARGETS_TO_TEST = None  # e.g. ['Outcome', 'Cue']
INTERVALS_TO_TEST = None  # e.g. ['R1_entry_interval', 'R2_exit_interval']
MAX_SLICES = None  # e.g. 120 for faster debug; None = all eligible slices
PROGRESS_EVERY = 20

# Keep these aligned with decoding cell
N_REL_BINS = 20
WINDOW_HALF_WIDTH = 1
MIN_SAMPLES_PER_BIN = 20
MIN_CLASS_COUNT = 3

EXCLUDED_INTERVALS = {
    'pre_cue_interval',
    'pre_cue_itnerval',
    'nextto_cue_interval',
}

OUTCOME_R1_POSITIVE_VALUES = {1}
OUTCOME_R2_POSITIVE_VALUES = {1, 2}

# Feature setup
ASSEMBLY_MODE = 'single'  # 'single' or 'all'
ASSEMBLY_COL = 'Assembly012'

def _bh_fdr(p_values):
    p = np.asarray(p_values, dtype=float)
    out = np.full_like(p, np.nan)
    mask = np.isfinite(p)
    if mask.sum() == 0:
        return out
    pv = p[mask]
    n = pv.size
    order = np.argsort(pv)
    ranked = pv[order]
    q = ranked * n / np.arange(1, n + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0.0, 1.0)
    q_full = np.empty_like(pv)
    q_full[order] = q
    out[mask] = q_full
    return out

def _build_model():
    return Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=1500, class_weight='balanced', solver='liblinear')),
        ]
    )

def _prepare_base_df():
    if 'ens_df' in globals() and isinstance(ens_df, pd.DataFrame):
        need_cols = {'session_id', 'interval_name', 'time_rel', 'time_rel_bin'}
        if need_cols.issubset(ens_df.columns):
            return ens_df.copy()

    if 't0_ens' not in globals() or t0_ens is None:
        if 'session_names' not in globals() or session_names is None:
            raise ValueError('Need either ens_df from prior cell or t0_ens + session_names in globals().')
        local_t0_ens = analytics.get_analytics('EnsembleT0Projection', session_names=session_names)
    else:
        local_t0_ens = t0_ens

    df = local_t0_ens.copy()
    if isinstance(df.index, pd.MultiIndex) or df.index.name is not None:
        df = df.reset_index()

    if 'interval_name' not in df.columns:
        raise ValueError('interval_name column not found in t0_ens')
    if 'session_id' not in df.columns:
        raise ValueError('session_id column not found in t0_ens')

    trial_col_local = next((c for c in ['trial_id', 'behavior_trial_id', 'trial', 'trial_index'] if c in df.columns), None)
    if trial_col_local is None:
        raise ValueError('No trial column found in t0_ens')

    df = df[~df['interval_name'].astype(str).isin(EXCLUDED_INTERVALS)].copy()

    sort_cols = ['session_id', trial_col_local, 'interval_name']
    for c in ['from_ephys_timestamp', 'to_ephys_timestamp', 'entry_id']:
        if c in df.columns:
            sort_cols.append(c)
    df = df.sort_values(sort_cols).copy()

    df['time_bin_idx'] = (
        df.groupby(['session_id', trial_col_local, 'interval_name'], sort=False, dropna=False)
        .cumcount()
        .astype(int)
    )
    df['n_bins_interval'] = (
        df.groupby(['session_id', trial_col_local, 'interval_name'], sort=False, dropna=False)['time_bin_idx']
        .transform('max')
        .add(1)
        .astype(int)
    )
    df['time_rel'] = np.where(
        df['n_bins_interval'] > 1,
        df['time_bin_idx'] / (df['n_bins_interval'] - 1),
        0.0,
    )
    df['time_rel_bin'] = np.clip(
        np.round(df['time_rel'] * (N_REL_BINS - 1)).astype(int),
        0,
        N_REL_BINS - 1,
    )

    for c in ['cue', 'choice_R1', 'choice_R2']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    if {'cue', 'choice_R1', 'choice_R2'}.issubset(df.columns):
        df['outcome_binary'] = (
            ((df['cue'] == 1) & (df['choice_R1'].isin(OUTCOME_R1_POSITIVE_VALUES)))
            | ((df['cue'] == 2) & (df['choice_R2'].isin(OUTCOME_R2_POSITIVE_VALUES)))
        ).astype(float)
    else:
        df['outcome_binary'] = np.nan

    return df

def _prepare_feature_cols(df):
    if 'feature_cols' in globals() and isinstance(feature_cols, list) and len(feature_cols) > 0:
        cols = [c for c in feature_cols if c in df.columns]
        if len(cols) > 0:
            return cols

    assembly_cols = [c for c in df.columns if str(c).startswith('Assembly')]
    if ASSEMBLY_MODE == 'single':
        if ASSEMBLY_COL not in df.columns:
            raise ValueError(f'{ASSEMBLY_COL} not found in t0_ens')
        return [ASSEMBLY_COL]
    if len(assembly_cols) == 0:
        raise ValueError('No Assembly* columns found')
    return assembly_cols

def _target_filter(df_t, target_col):
    out = df_t.copy()
    out[target_col] = pd.to_numeric(out[target_col], errors='coerce')
    out = out.dropna(subset=[target_col])
    if target_col == 'cue':
        out = out[out[target_col].isin([1, 2])]
    else:
        out = out[out[target_col].isin([0, 1])]
    out[target_col] = out[target_col].astype(int)
    return out

def _build_trial_bin_df(df, session_col, trial_col, interval_col, feature_cols_local, target_col):
    cols = [session_col, trial_col, interval_col, 'time_rel', 'time_rel_bin', *feature_cols_local, target_col]
    out = df[cols].copy()
    out = out.dropna(subset=[session_col, trial_col, interval_col, 'time_rel', 'time_rel_bin', target_col, *feature_cols_local])
    out = _target_filter(out, target_col)
    if out.empty:
        return out

    agg_map = {c: 'mean' for c in feature_cols_local}
    agg_map['time_rel'] = 'mean'
    agg_map[target_col] = 'first'

    return (
        out.groupby([session_col, trial_col, interval_col, 'time_rel_bin'], as_index=False)
        .agg(agg_map)
    )

def _window_pool(interval_df, center_bin, window_half_width, session_col, trial_col, feature_cols_local, target_col):
    lo = center_bin - window_half_width
    hi = center_bin + window_half_width
    win = interval_df[interval_df['time_rel_bin'].between(lo, hi)].copy()
    if win.empty:
        return win

    pooled = (
        win.groupby([session_col, trial_col], as_index=False)
        .agg({**{c: 'mean' for c in feature_cols_local}, target_col: 'first', 'time_rel': 'mean'})
    )
    pooled['trial_uid'] = pooled[session_col].astype(str) + '__' + pooled[trial_col].astype(str)
    return pooled

def _build_splits(y, session_groups, trial_groups, cv_mode, random_state, max_cv_splits):
    cls_counts = pd.Series(y).value_counts(dropna=True)
    if len(cls_counts) < 2:
        return None, 'single_class', 0
    if int(cls_counts.min()) < MIN_CLASS_COUNT:
        return None, 'too_few_per_class', 0

    if cv_mode == 'loto':
        g = pd.Series(trial_groups).astype(str).to_numpy()
        if len(np.unique(g)) < 3:
            return None, 'too_few_trials', 0
        cv = LeaveOneGroupOut()
        splits = list(cv.split(np.zeros(len(y)), y, groups=g))
        return splits, 'loto', len(splits)

    g = pd.Series(session_groups).astype(str).to_numpy()
    n_groups = len(np.unique(g))
    if n_groups < 3:
        return None, 'too_few_sessions', 0
    n_splits = int(min(max_cv_splits, n_groups))
    if n_splits < 2:
        return None, 'cv_not_possible', 0

    try:
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        splits = list(cv.split(np.zeros(len(y)), y, groups=g))
        return splits, 'stratified_group', len(splits)
    except ValueError:
        return None, 'cv_error', 0

def _oof_predictions(X, y, splits):
    y_pred = np.empty_like(y)
    for tr_idx, te_idx in splits:
        model = _build_model()
        model.fit(X[tr_idx], y[tr_idx])
        y_pred[te_idx] = model.predict(X[te_idx])
    return y_pred

def _permute_within_sessions(y, session_arr, rng):
    yp = y.copy()
    sess = np.asarray(session_arr)
    for s in np.unique(sess):
        idx = np.flatnonzero(sess == s)
        if len(idx) > 1:
            yp[idx] = yp[rng.permutation(idx)]
    return yp

# Build slice table
base_df = _prepare_base_df()
session_col = 'session_id'
trial_col = next((c for c in ['trial_id', 'behavior_trial_id', 'trial', 'trial_index'] if c in base_df.columns), None)
interval_col = 'interval_name'
feature_cols_local = _prepare_feature_cols(base_df)

for c in feature_cols_local:
    base_df[c] = pd.to_numeric(base_df[c], errors='coerce')

print('Permutation-test setup:')
print(f'  cv_mode={CV_MODE} perm_mode={PERM_MODE} n_perm={N_PERMUTATIONS}')
print(f'  n_features={len(feature_cols_local)} | features={feature_cols_local[:8]}{" ..." if len(feature_cols_local) > 8 else ""}')
print(f'  excluded_intervals={sorted(EXCLUDED_INTERVALS)}')

all_targets = {
    'Outcome': 'outcome_binary',
    'Cue': 'cue',
    'R1 choice': 'choice_R1',
    'R2 choice': 'choice_R2',
}
if TARGETS_TO_TEST is not None:
    target_map = {k: v for k, v in all_targets.items() if k in set(TARGETS_TO_TEST)}
else:
    target_map = all_targets

slice_specs = []

for target_name, target_col in target_map.items():
    if target_col not in base_df.columns:
        continue

    trial_bin_df = _build_trial_bin_df(
        base_df,
        session_col=session_col,
        trial_col=trial_col,
        interval_col=interval_col,
        feature_cols_local=feature_cols_local,
        target_col=target_col,
    )
    if trial_bin_df.empty:
        continue

    for interval_name, interval_df in trial_bin_df.groupby(interval_col):
        if INTERVALS_TO_TEST is not None and interval_name not in set(INTERVALS_TO_TEST):
            continue

        for b in sorted(interval_df['time_rel_bin'].unique()):
            pooled = _window_pool(
                interval_df,
                center_bin=int(b),
                window_half_width=WINDOW_HALF_WIDTH,
                session_col=session_col,
                trial_col=trial_col,
                feature_cols_local=feature_cols_local,
                target_col=target_col,
            )
            if pooled.empty:
                continue

            n_samples = int(len(pooled))
            cls_counts = pooled[target_col].value_counts(dropna=True)
            n_classes = int(len(cls_counts))
            min_class = int(cls_counts.min()) if n_classes else 0

            if n_samples < MIN_SAMPLES_PER_BIN:
                continue
            if n_classes != 2:
                continue
            if min_class < MIN_CLASS_COUNT:
                continue

            slice_specs.append(
                {
                    'target_name': target_name,
                    'target_col': target_col,
                    'interval_name': str(interval_name),
                    'time_rel_bin': int(b),
                    'time_rel': float(pooled['time_rel'].median()),
                    'n_samples': n_samples,
                    'n_trials': int(pooled['trial_uid'].nunique()),
                    'n_sessions': int(pooled[session_col].astype(str).nunique()),
                    'pooled_df': pooled,
                }
            )

if MAX_SLICES is not None:
    slice_specs = slice_specs[: int(MAX_SLICES)]

print(f'Eligible slices for permutation test: {len(slice_specs)}')
if len(slice_specs) == 0:
    raise ValueError('No eligible slices found. Relax filters or inspect class balance.')

# Observed metrics + permutation null
rng = np.random.default_rng(RANDOM_STATE)
rows = []
started = time.perf_counter()

for i, spec in enumerate(slice_specs, start=1):
    pooled = spec['pooled_df']
    target_col = spec['target_col']

    X = pooled[feature_cols_local].to_numpy(dtype=float)
    y = pooled[target_col].to_numpy(dtype=int)
    sess = pooled[session_col].astype(str).to_numpy()
    trial_uid = pooled['trial_uid'].astype(str).to_numpy()

    splits, cv_strategy, n_folds = _build_splits(
        y=y,
        session_groups=sess,
        trial_groups=trial_uid,
        cv_mode=CV_MODE,
        random_state=RANDOM_STATE,
        max_cv_splits=MAX_CV_SPLITS,
    )

    if not splits:
        rows.append(
            {
                **{k: v for k, v in spec.items() if k != 'pooled_df'},
                'cv_strategy': cv_strategy,
                'n_folds': int(n_folds),
                'obs_acc': np.nan,
                'obs_bal_acc': np.nan,
                'null_acc_mean': np.nan,
                'null_acc_std': np.nan,
                'null_bal_acc_mean': np.nan,
                'null_bal_acc_std': np.nan,
                'p_acc': np.nan,
                'p_bal_acc': np.nan,
                'n_permutations': int(N_PERMUTATIONS),
            }
        )
        continue

    try:
        y_pred = _oof_predictions(X, y, splits)
        obs_acc = float(accuracy_score(y, y_pred))
        obs_bal_acc = float(balanced_accuracy_score(y, y_pred))
    except Exception:
        rows.append(
            {
                **{k: v for k, v in spec.items() if k != 'pooled_df'},
                'cv_strategy': 'fit_error',
                'n_folds': int(n_folds),
                'obs_acc': np.nan,
                'obs_bal_acc': np.nan,
                'null_acc_mean': np.nan,
                'null_acc_std': np.nan,
                'null_bal_acc_mean': np.nan,
                'null_bal_acc_std': np.nan,
                'p_acc': np.nan,
                'p_bal_acc': np.nan,
                'n_permutations': int(N_PERMUTATIONS),
            }
        )
        continue

    null_acc = np.full(N_PERMUTATIONS, np.nan, dtype=float)
    null_bal_acc = np.full(N_PERMUTATIONS, np.nan, dtype=float)

    for pidx in range(N_PERMUTATIONS):
        y_perm = _permute_within_sessions(y, sess, rng)

        if PERM_MODE == 'conditional':
            y_perm_pred = y_pred
        else:
            try:
                y_perm_pred = _oof_predictions(X, y_perm, splits)
            except Exception:
                continue

        null_acc[pidx] = float(accuracy_score(y_perm, y_perm_pred))
        null_bal_acc[pidx] = float(balanced_accuracy_score(y_perm, y_perm_pred))

    valid_null_acc = null_acc[np.isfinite(null_acc)]
    valid_null_bal = null_bal_acc[np.isfinite(null_bal_acc)]

    if len(valid_null_acc) == 0 or len(valid_null_bal) == 0:
        p_acc = np.nan
        p_bal = np.nan
        null_acc_mean = np.nan
        null_acc_std = np.nan
        null_bal_mean = np.nan
        null_bal_std = np.nan
    else:
        p_acc = float((1 + np.sum(valid_null_acc >= obs_acc)) / (len(valid_null_acc) + 1))
        p_bal = float((1 + np.sum(valid_null_bal >= obs_bal_acc)) / (len(valid_null_bal) + 1))
        null_acc_mean = float(np.mean(valid_null_acc))
        null_acc_std = float(np.std(valid_null_acc, ddof=1)) if len(valid_null_acc) > 1 else 0.0
        null_bal_mean = float(np.mean(valid_null_bal))
        null_bal_std = float(np.std(valid_null_bal, ddof=1)) if len(valid_null_bal) > 1 else 0.0

    rows.append(
        {
            **{k: v for k, v in spec.items() if k != 'pooled_df'},
            'cv_strategy': cv_strategy,
            'n_folds': int(n_folds),
            'obs_acc': obs_acc,
            'obs_bal_acc': obs_bal_acc,
            'null_acc_mean': null_acc_mean,
            'null_acc_std': null_acc_std,
            'null_bal_acc_mean': null_bal_mean,
            'null_bal_acc_std': null_bal_std,
            'p_acc': p_acc,
            'p_bal_acc': p_bal,
            'n_permutations': int(len(valid_null_bal)),
        }
    )

    if i == 1 or i % max(1, int(PROGRESS_EVERY)) == 0:
        elapsed = time.perf_counter() - started
        print(f'  progress: slice {i}/{len(slice_specs)} elapsed_s={elapsed:.1f}')

perm_significance_df = pd.DataFrame(rows)

# Multiple-testing correction
if not perm_significance_df.empty:
    perm_significance_df['q_bal_acc_global'] = _bh_fdr(perm_significance_df['p_bal_acc'].to_numpy())
    perm_significance_df['q_acc_global'] = _bh_fdr(perm_significance_df['p_acc'].to_numpy())

    perm_significance_df['q_bal_acc_target'] = np.nan
    perm_significance_df['q_acc_target'] = np.nan
    for tname, sub_idx in perm_significance_df.groupby('target_name').groups.items():
        idx = np.array(list(sub_idx))
        perm_significance_df.loc[idx, 'q_bal_acc_target'] = _bh_fdr(perm_significance_df.loc[idx, 'p_bal_acc'].to_numpy())
        perm_significance_df.loc[idx, 'q_acc_target'] = _bh_fdr(perm_significance_df.loc[idx, 'p_acc'].to_numpy())

print('Done permutation significance test')
print(f'Rows in perm_significance_df: {len(perm_significance_df)}')

if not perm_significance_df.empty:
    print()
    print('Significant bins summary (balanced accuracy):')
    sig_summary = (
        perm_significance_df.assign(
            sig_global=perm_significance_df['q_bal_acc_global'] < 0.05,
            sig_target=perm_significance_df['q_bal_acc_target'] < 0.05,
        )
        .groupby('target_name', as_index=False)
        .agg(
            n_bins=('target_name', 'size'),
            n_sig_global=('sig_global', 'sum'),
            n_sig_target=('sig_target', 'sum'),
            mean_obs_bal_acc=('obs_bal_acc', 'mean'),
            mean_null_bal_acc=('null_bal_acc_mean', 'mean'),
        )
    )
    display(sig_summary)

    print()
    print('Top significant bins by q_bal_acc_global:')
    cols_show = [
        'target_name', 'interval_name', 'time_rel_bin', 'time_rel',
        'n_samples', 'obs_bal_acc', 'null_bal_acc_mean', 'p_bal_acc', 'q_bal_acc_global'
    ]
    display(
        perm_significance_df.sort_values('q_bal_acc_global', na_position='last')
        .loc[:, cols_show]
        .head(30)
    )

# Plot with significance markers
if not perm_significance_df.empty:
    for target_name in perm_significance_df['target_name'].dropna().unique():
        sub = perm_significance_df[
            (perm_significance_df['target_name'] == target_name)
            & perm_significance_df['obs_bal_acc'].notna()
        ].sort_values(['interval_name', 'time_rel_bin'])

        if sub.empty:
            continue

        fig = px.line(
            sub,
            x='time_rel',
            y='obs_bal_acc',
            color='interval_name',
            markers=True,
            title=f'Permutation test (balanced accuracy): {target_name}',
            labels={
                'time_rel': 'Relative time in interval (0=start, 1=end)',
                'obs_bal_acc': 'Observed balanced accuracy',
                'interval_name': 'Interval',
            },
            hover_data=['n_samples', 'p_bal_acc', 'q_bal_acc_global', 'q_bal_acc_target'],
        )

        sig = sub[sub['q_bal_acc_global'] < 0.05]
        if not sig.empty:
            fig.add_scatter(
                x=sig['time_rel'],
                y=sig['obs_bal_acc'],
                mode='markers',
                marker=dict(color='black', size=8, symbol='x'),
                name='q<0.05 (global FDR)',
                hovertemplate='sig bin<extra></extra>',
            )

        fig.update_layout(template='plotly_white')
        fig.show()

Permutation-test setup:
  cv_mode=loto perm_mode=conditional n_perm=500
  n_features=1 | features=['Assembly012']
  excluded_intervals=['nextto_cue_interval', 'pre_cue_interval', 'pre_cue_itnerval']
Eligible slices for permutation test: 520
  progress: slice 1/520 elapsed_s=7.0
  progress: slice 20/520 elapsed_s=141.9
  progress: slice 40/520 elapsed_s=283.9
  progress: slice 60/520 elapsed_s=425.4
  progress: slice 80/520 elapsed_s=568.9
  progress: slice 100/520 elapsed_s=709.5
  progress: slice 120/520 elapsed_s=850.1
  progress: slice 140/520 elapsed_s=998.3
  progress: slice 160/520 elapsed_s=1144.5
  progress: slice 180/520 elapsed_s=1291.3
  progress: slice 200/520 elapsed_s=1438.4
  progress: slice 220/520 elapsed_s=1584.8
  progress: slice 240/520 elapsed_s=1731.0
  progress: slice 260/520 elapsed_s=1874.1
  progress: slice 280/520 elapsed_s=2017.2
  progress: slice 300/520 elapsed_s=2160.6
  progress: slice 320/520 elapsed_s=2303.7
  progress: slice 340/520 elapsed_s=2446.8
 

,target_name,n_bins,n_sig_global,n_sig_target,mean_obs_bal_acc,mean_null_bal_acc
0,Cue,120,72,79,0.523997,0.500951
1,Outcome,120,47,51,0.516289,0.505086
2,R1 choice,140,15,13,0.515184,0.510839
3,R2 choice,140,22,20,0.532226,0.533865



Top significant bins by q_bal_acc_global:


,target_name,interval_name,time_rel_bin,time_rel,n_samples,obs_bal_acc,null_bal_acc_mean,p_bal_acc,q_bal_acc_global
32,Outcome,R1_exit_interval,12,0.628205,2716,0.560862,0.528299,0.001996,0.009351
33,Outcome,R1_exit_interval,13,0.679487,2716,0.546813,0.522205,0.001996,0.009351
31,Outcome,R1_exit_interval,11,0.576923,2716,0.566283,0.528224,0.001996,0.009351
30,Outcome,R1_exit_interval,10,0.525641,2716,0.558906,0.521028,0.001996,0.009351
29,Outcome,R1_exit_interval,9,0.474359,2716,0.561504,0.523031,0.001996,0.009351
28,Outcome,R1_exit_interval,8,0.423077,2716,0.560959,0.521079,0.001996,0.009351
27,Outcome,R1_exit_interval,7,0.371795,2716,0.544599,0.516018,0.001996,0.009351
456,R2 choice,R2_exit_interval,16,0.833333,2716,0.583343,0.563257,0.001996,0.009351
457,R2 choice,R2_exit_interval,17,0.884615,2716,0.587737,0.566187,0.001996,0.009351
458,R2 choice,R2_exit_interval,18,0.935897,2716,0.583812,0.563541,0.001996,0.009351
